# Chapter 12 — Preserving Provenance · Practice

**Goal:** keep every claim tied to its evidence from the first agent that reads a source to the last line of the report: claim records, conflict and date handling, content-type rendering, coverage reporting, the Citations API, `search_result` blocks, web citations and a provenance audit, ending with a small multi-source research synthesizer.

- Theory: [README.md](README.md) (read section 12.x before running the matching section here)
- Estimated time: 3–4 hours
- Estimated API cost: **about $0.05–0.15** with the default model (`claude-sonnet-5-5`, every live cell once), plus about $0.02–0.05 for the optional web search cell (up to 2 searches at $10 per 1,000, plus the search results as input tokens). This is an **estimate, not a measurement**: it was computed from the list prices on the [pricing page](https://platform.claude.com/docs/en/about-claude/pricing) as of 2026-10-03 (Claude Sonnet 5.5: $2 input / $10 output per million tokens) and the cells' `max_tokens` caps; your real cost depends on how long the answers are. Check `response.usage` after each live cell. Every section also has **offline** cells, so you can learn the mechanics with no key and no credit.
- Prerequisites: [00-prerequisites](../../00-prerequisites/README.md) (venv from `requirements.txt`, `.env` with `ANTHROPIC_API_KEY`), [Chapter 2](../02-tool-use/README.md) (tool loop, server tools), [Chapter 6](../06-prompt-engineering/README.md) (structured outputs), [Chapter 10](../10-multi-agent-error-handling/README.md) (structured error context, coverage)

How to read the cells:

| Marker | Meaning |
|---|---|
| **(offline)** | Runs without a key. Uses `ScriptedClaude`, a real `anthropic.Anthropic` client whose HTTP layer replays recorded responses and records what you sent, so the SDK's own types (`CitationCharLocation`, `WebSearchToolResultBlock`, ...) are parsed for real. |
| **(live)** | Calls the real API and costs a fraction of a cent. Skip it if you have no key; nothing offline depends on it. |

All sources in this notebook are **fictional fixtures** (example.com domains, invented organizations and numbers) built to exercise the patterns. Do not quote them as facts about the music industry.

In [ ]:
import json
import os
import re
import tempfile
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime
from itertools import count
from pathlib import Path
from typing import Any, Callable, Literal

import anthropic
import httpx2
from dotenv import find_dotenv, load_dotenv
from IPython.display import Markdown, display
from pydantic import BaseModel, Field, ValidationError

load_dotenv(find_dotenv())

MODEL = os.getenv("CLAUDE_MODEL", "claude-sonnet-5-5")
AS_OF = date(2026, 10, 3)  # a fixed "today" so outputs are reproducible; use date.today() in production

client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY; only the (live) cells use it
print("anthropic", anthropic.__version__, "| MODEL =", MODEL, "| AS_OF =", AS_OF)

In [ ]:
# Live cells need a key. Without one, every (offline) cell still runs; only the (live) cells fail.
if not os.getenv("ANTHROPIC_API_KEY"):
    print("No ANTHROPIC_API_KEY: the (live) cells will fail; every (offline) cell works. See 00-prerequisites.")

### Offline toolkit (run this once)

`ScriptedClaude` gives you a **real** SDK client (`.client`) whose transport answers with the responses you script, in order, and stores every request body in `.requests`. A scripted reply can be a JSON body (normal responses), an error, or a raw server-sent-events string (to test streaming). The `*_cite()` helpers build citation objects in the exact wire format, computing character offsets from the real fixture text so that the integrity checks later on are meaningful.

In [ ]:
_ids = count(1)


def text(t: str, citations: list[dict] | None = None) -> dict:
    block = {"type": "text", "text": t}
    if citations is not None:
        block["citations"] = citations
    return block


def tool_call(name: str, tool_input: dict, id: str | None = None) -> dict:
    return {"type": "tool_use", "id": id or f"toolu_{next(_ids):04d}", "name": name, "input": tool_input}


def reply(*blocks: dict, stop_reason: str | None = None) -> dict:
    """A Messages API response body. stop_reason defaults to tool_use/end_turn based on the blocks."""
    if stop_reason is None:
        stop_reason = "tool_use" if any(b["type"] == "tool_use" for b in blocks) else "end_turn"
    return {"id": f"msg_{next(_ids):04d}", "type": "message", "role": "assistant", "model": "scripted",
            "content": list(blocks), "stop_reason": stop_reason, "stop_sequence": None,
            "usage": {"input_tokens": 0, "output_tokens": 0}}


def json_reply(obj: Any) -> dict:
    """What a structured-outputs response looks like: one text block holding JSON."""
    return reply(text(json.dumps(obj)))


class ScriptedClaude:
    """Offline stand-in for the Messages API: replays scripted replies, records every request."""

    def __init__(self, *replies: dict | str):
        self.replies = list(replies)
        self.requests: list[dict] = []
        transport = httpx2.MockTransport(self._handle)
        self.client = anthropic.Anthropic(
            api_key="offline", max_retries=0, http_client=anthropic.DefaultHttpxClient(transport=transport))

    def _handle(self, request: httpx2.Request) -> httpx2.Response:
        self.requests.append(json.loads(request.content))
        if not self.replies:
            return httpx2.Response(500, json={"type": "error", "error": {"type": "api_error", "message": "script exhausted"}})
        body = self.replies.pop(0)
        if isinstance(body, str):  # a recorded SSE stream
            return httpx2.Response(200, headers={"content-type": "text/event-stream"}, content=body.encode())
        body = dict(body)
        return httpx2.Response(body.pop("_status", 200), json=body)


# ---- citation builders (wire format) ----
def char_cite(docs: list[dict], doc_index: int, sentence: str) -> dict:
    data = docs[doc_index]["text"]
    start = data.index(sentence)  # raises if the "quote" is not really in the document
    return {"type": "char_location", "cited_text": sentence, "document_index": doc_index,
            "document_title": docs[doc_index]["title"], "start_char_index": start, "end_char_index": start + len(sentence)}


def page_cite(doc_index: int, title: str, cited_text: str, start_page: int, end_page_exclusive: int) -> dict:
    return {"type": "page_location", "cited_text": cited_text, "document_index": doc_index, "document_title": title,
            "start_page_number": start_page, "end_page_number": end_page_exclusive}


def block_cite(doc_index: int, title: str, blocks: list[str], start: int, end: int) -> dict:
    return {"type": "content_block_location", "cited_text": " ".join(blocks[start:end]), "document_index": doc_index,
            "document_title": title, "start_block_index": start, "end_block_index": end}


def sr_cite(result_index: int, result: dict, start: int, end: int) -> dict:
    texts = [b["text"] for b in result["content"]]
    return {"type": "search_result_location", "cited_text": " ".join(texts[start:end]), "source": result["source"],
            "title": result["title"], "search_result_index": result_index, "start_block_index": start, "end_block_index": end}


def web_cite(url: str, title: str, cited_text: str) -> dict:
    return {"type": "web_search_result_location", "url": url, "title": title,
            "encrypted_index": f"enc-idx-{next(_ids)}", "cited_text": cited_text[:150]}


def show(response, width: int = 100) -> None:
    """Print stop_reason, each content block (with its citations) and usage."""
    print("stop_reason:", response.stop_reason)
    for b in response.content:
        if b.type == "text":
            print(f"  [text] {b.text[:width]!r}")
            for c in b.citations or []:
                where = {"char_location": lambda: f"doc {c.document_index} chars {c.start_char_index}-{c.end_char_index}",
                         "page_location": lambda: f"doc {c.document_index} pages {c.start_page_number}-{c.end_page_number}",
                         "content_block_location": lambda: f"doc {c.document_index} blocks {c.start_block_index}-{c.end_block_index}",
                         "search_result_location": lambda: f"{c.source} blocks {c.start_block_index}-{c.end_block_index}",
                         "web_search_result_location": lambda: c.url}[c.type]()
                print(f"      ↳ {c.type}: {where} | {c.cited_text[:60]!r}")
        elif b.type == "server_tool_use":
            print(f"  [server_tool_use] {b.name}({json.dumps(b.input)})")
        elif b.type == "web_search_tool_result":
            items = b.content if isinstance(b.content, list) else b.content.error_code
            print(f"  [web_search_tool_result] {len(items) if isinstance(items, list) else 'ERROR ' + items}")
        else:
            print(f"  [{b.type}]")
    print(f"usage: in={response.usage.input_tokens} out={response.usage.output_tokens}")


print("offline toolkit ready")

### The fixture sources

One research question runs through the whole notebook: *"How big is the AI-generated music market, and how reliable are the numbers?"* Five sources answer it. Two credible statistics disagree about 2024 (and agree about 2023), one news article reports industry deals, one technical note reports detection accuracy, and the fifth source (a news and social-media archive) will **time out**.

In [ ]:
FIXTURES = [
    {"source_id": "S1", "title": "Global AI Music Market Report 2024", "publisher": "Example Analytics",
     "url": "https://example.com/ai-music-report-2024", "source_type": "industry_report", "published": "2024-06-15",
     "retrieved_at": "2026-10-01", "methodology": "analyst estimate (top-down model)", "derived_from": None,
     "text": ("Global AI Music Market Report 2024. Executive summary. "
              "We estimate the global market for AI-generated music at USD 3.2 billion in 2024, up from USD 1.9 billion in 2023. "
              "Our top-down model combines streaming revenue shares with licensing fees paid to AI music providers. "
              "We project the market to reach USD 7.5 billion by 2027. "
              "These figures are estimates with a margin of error of plus or minus 15%.")},
    {"source_id": "S2", "title": "Creative Economy Statistics 2025", "publisher": "Example National Statistics Office",
     "url": "https://stats.example.gov/creative-economy-2025", "source_type": "government", "published": "2025-02-20",
     "retrieved_at": "2026-10-01", "methodology": "bottom-up count of revenue declared in tax filings", "derived_from": None,
     "text": ("Creative Economy Statistics 2025. Section 4: AI-generated content. "
              "Revenue reported by registered firms selling AI-generated music totalled USD 2.1 billion in 2024, compared with USD 1.9 billion in 2023. "
              "The figures are a bottom-up count of revenue declared in tax filings. "
              "Figures for 2024 are preliminary and will be revised in the final release. "
              "Unregistered and foreign platforms are not included.")},
    {"source_id": "S3", "title": "Major labels sign licensing deals with AI music start-ups", "publisher": "Example News",
     "url": "https://news.example.com/labels-ai-licensing", "source_type": "news", "published": "2024-09-02",
     "retrieved_at": "2026-10-01", "methodology": None, "derived_from": None,
     "text": ("Major labels sign licensing deals with AI music start-ups. "
              "Three major record labels announced licensing agreements with AI music start-ups this week. "
              "The deals let the start-ups train models on licensed catalogues in exchange for royalties. "
              "Label executives said the agreements were preliminary and could change.")},
    {"source_id": "S4", "title": "Detecting AI-generated audio at scale (TN-24-07)", "publisher": "Example University Audio Lab",
     "url": "https://lab.example.edu/tn-24-07", "source_type": "academic", "published": "2024-05-20",
     "retrieved_at": "2026-10-01", "methodology": "benchmark evaluation on 10,000 tracks", "derived_from": None,
     "text": ("Detecting AI-generated audio at scale. Technical note TN-24-07. "
              "Our classifier reached 94% precision and 81% recall on a benchmark of 10,000 tracks. "
              "Recall drops to 62% for tracks that mix human vocals with generated instrumentals. "
              "False positives were concentrated in heavily auto-tuned pop recordings. "
              "We recommend human review for any track scored between 0.4 and 0.6.")},
    {"source_id": "S5", "title": "News and social media archive", "publisher": "Example Archive Service",
     "url": "https://archive.example.org/search", "source_type": "other", "published": None,
     "retrieved_at": None, "methodology": None, "derived_from": None, "text": None},   # this one times out
]
FIXTURE_BY_ID = {f["source_id"]: f for f in FIXTURES}
DOCS = [f for f in FIXTURES if f["text"]]          # S1..S4: the readable ones
for f in DOCS:
    print(f"{f['source_id']}  {f['published']}  {f['source_type']:16} {f['title']}")

## 12.1 The attribution loss problem

Every summarizing step keeps the *what* and drops the *who, when and how*: sources, verbatim excerpts, qualifiers ("estimated", "preliminary"), dates and methods disappear because they look like noise. The fix is a **claim record** (statement + verbatim excerpt + source + location + dates + method) that every agent passes on as data, with required-but-nullable fields so that missing metadata is `null`, never invented. [Theory](README.md#121-the-attribution-loss-problem)

### Measure attribution loss across a three-hop chain (offline)

We track four numeric facts and ask, after each summarizing hop: is the number still there, is a source still named next to it, and is its qualifier still there? The scripted "summaries" are typical of what real models produce when told to be brief.

In [ ]:
TRACKED = [
    {"fact": "2024 market (analyst)", "number": "3.2", "markers": ["Example Analytics", "Market Report"], "qualifiers": ["estimat"]},
    {"fact": "2024 market (official)", "number": "2.1", "markers": ["Statistics Office", "official statistics"], "qualifiers": ["preliminary"]},
    {"fact": "2027 forecast", "number": "7.5", "markers": ["Example Analytics", "Market Report"], "qualifiers": ["project"]},
    {"fact": "detector recall", "number": "81%", "markers": ["Audio Lab", "TN-24-07", "technical note"], "qualifiers": ["benchmark"]},
]


def passage_with(text: str, needle: str) -> str:
    """The paragraph that contains `needle`: attribution has to be near the number to count."""
    return next((p for p in text.split("\n\n") if needle in p), "")


def attribution_report(text: str) -> dict:
    kept = [f for f in TRACKED if f["number"] in text]
    sourced = [f for f in kept if any(m.lower() in passage_with(text, f["number"]).lower() for m in f["markers"])]
    hedged = [f for f in kept if any(q in passage_with(text, f["number"]).lower() for q in f["qualifiers"])]
    return {"numbers": len(kept), "with_source": len(sourced), "with_qualifier": len(hedged)}


def as_context(docs: list[dict]) -> str:
    return "\n\n".join(f"[{d['source_id']}] {d['title']} ({d['publisher']}, published {d['published']})\n{d['text']}" for d in docs)


def summarize_chain(llm: anthropic.Anthropic, text: str, hops: int = 3) -> list[str]:
    """Each hop summarizes the previous hop's output, the way a naive multi-agent pipeline does."""
    outputs = [text]
    for _ in range(hops):
        r = llm.messages.create(model=MODEL, max_tokens=300, messages=[{"role": "user", "content":
            "Summarize the following for a busy executive in at most 3 sentences.\n\n" + outputs[-1]}])
        outputs.append("".join(b.text for b in r.content if b.type == "text"))
    return outputs


def print_loss_table(outputs: list[str]) -> None:
    print(f"{'hop':4} {'numbers':8} {'with source':12} {'with qualifier':15} text")
    for i, out in enumerate(outputs):
        r = attribution_report(out)
        print(f"{i:<4} {r['numbers']:<8} {r['with_source']:<12} {r['with_qualifier']:<15} {out[:70].replace(chr(10), ' ')!r}")


fake = ScriptedClaude(
    reply(text("Example Analytics' Global AI Music Market Report estimates the AI music market at USD 3.2 billion in 2024 "
               "and projects USD 7.5 billion by 2027. The national statistics office reports a preliminary USD 2.1 billion. "
               "An audio lab technical note reports 81% recall for its detector on a benchmark.")),
    reply(text("The AI music market is about USD 3.2 billion and will reach USD 7.5 billion by 2027, although official "
               "data show USD 2.1 billion. Detection tools catch 81% of AI tracks.")),
    reply(text("The AI music market is worth USD 3.2 billion and will reach USD 7.5 billion by 2027; detectors catch 81% of AI tracks.")),
)
print_loss_table(summarize_chain(fake.client, as_context(DOCS)))

> **Observe:** hop 1 still names its sources and hedges. Hop 2 keeps every number but drops every source and every qualifier, and by hop 3 the official figure (the one that disagrees) is gone entirely. "Estimated at" became "is worth", and "projects" became "will reach". No hop *decided* to drop provenance; each just optimized for brevity. (The metric is deliberately simple: a source name anywhere in the same paragraph counts.)

> **Try it:** change the prompt to "Summarize ... and keep every source name and qualifier" and script what you think a model would return. A prompt helps, but nothing *enforces* it: that is why the fix is a data contract, not an instruction.

In [ ]:
# (live) the same three-hop chain on the real model
print_loss_table(summarize_chain(client, as_context(DOCS)))

### The claim record contract (offline)

Based on the README's Pydantic models (12.1), plus `publisher` and `source_type` on `Source` and without the optional `confidence` field on `Claim`. `Field(min_length=1)` on `evidence` makes "no evidence, no claim" a schema rule, and `anthropic.transform_schema` shows what `messages.parse()` sends: every field is required, and nullable fields become `anyOf [..., null]`.

In [ ]:
class Source(BaseModel):
    source_id: str
    title: str
    url: str | None
    publisher: str | None
    source_type: Literal["government", "industry_report", "academic", "news", "company_filing", "internal", "other"]
    published: str | None          # ISO 8601; null if unknown, never guessed
    retrieved_at: str | None
    methodology: str | None
    derived_from: str | None       # source_id of the original if this source only re-reports it


class Evidence(BaseModel):
    source_id: str
    excerpt: str                   # verbatim
    location: str | None


class Claim(BaseModel):
    claim_id: str
    statement: str                 # keeps the source's qualifiers
    subject: str
    metric: str | None
    value: float | None
    unit: str | None
    period: str | None             # the period the value describes
    evidence: list[Evidence] = Field(min_length=1)


class ClaimBatch(BaseModel):
    claims: list[Claim]


SOURCES: dict[str, Source] = {f["source_id"]: Source(**{k: v for k, v in f.items() if k != "text"}) for f in FIXTURES}

schema = anthropic.transform_schema(ClaimBatch)
claim_schema = schema["$defs"]["Claim"]
print("Claim.required:", claim_schema["required"])
print("period:", claim_schema["properties"]["period"])
print("evidence minItems:", claim_schema["properties"]["evidence"].get("minItems"))

try:
    Claim(claim_id="C0", statement="The market is huge.", subject="AI music", metric=None, value=None,
          unit=None, period=None, evidence=[])
except ValidationError as e:
    print("\nrejected:", e.errors()[0]["msg"])

### A subagent that returns claim records (offline)

A subagent prompt built from the 12.1 instructions, plus few-shot guidance for sources that cite their *own* sources (inline "(Author, Year)" vs a numbered bibliography). The scripted model returns one claim whose excerpt is a paraphrase, not a quote; `excerpt_problems()` catches it, because a verbatim excerpt must be a substring of the source text.

In [ ]:
SUBAGENT_PROMPT = """You extract findings from the documents you are given as claim records.
For every finding, output one claim record:
- statement: one sentence; keep the source's qualifiers (estimated, preliminary, projected, up to).
- evidence: a verbatim excerpt (copy the words exactly), the source_id, and the page or section.
- period: the time period the number describes. Use null when the source does not state it.
  Never infer a date from your own knowledge.
- One finding per record. If two sources state the same finding, list both under evidence.
Do not summarize findings without their evidence. A finding without a source is not reported.

How to handle a source's own citations:
<example>Text: "Streaming revenue grew 9% (Lee, 2023)." -> evidence.excerpt keeps "(Lee, 2023)";
statement: "Streaming revenue grew 9%, according to Lee (2023) as cited in <source title>."</example>
<example>Text: "Adoption reached 40% [3]." with reference list "[3] Music Survey 2024" ->
statement: "Adoption reached 40%, according to Music Survey 2024 as cited in <source title>."</example>"""


def excerpt_problems(claims: list[Claim], texts: dict[str, str]) -> list[str]:
    """Every excerpt must literally appear in the stored source text (whitespace-normalized)."""
    norm = lambda s: " ".join(s.split())
    problems = []
    for c in claims:
        for e in c.evidence:
            if e.source_id not in texts:
                problems.append(f"{c.claim_id}: unknown source {e.source_id}")
            elif norm(e.excerpt) not in norm(texts[e.source_id]):
                problems.append(f"{c.claim_id}: excerpt not found verbatim in {e.source_id}: {e.excerpt[:60]!r}")
    return problems


TEXTS = {d["source_id"]: d["text"] for d in DOCS}
scripted_batch = {"claims": [
    {"claim_id": "C1", "statement": "Revenue reported by registered firms was a preliminary USD 2.1 billion in 2024.",
     "subject": "AI-generated music market", "metric": "market size", "value": 2.1, "unit": "USD bn", "period": "2024",
     "evidence": [{"source_id": "S2", "location": "Section 4",
                   "excerpt": "Revenue reported by registered firms selling AI-generated music totalled USD 2.1 billion in 2024"}]},
    {"claim_id": "C2", "statement": "Foreign platforms are excluded from the official count.",
     "subject": "AI-generated music market", "metric": None, "value": None, "unit": None, "period": None,
     "evidence": [{"source_id": "S2", "location": "Section 4",
                   "excerpt": "The statistics leave out foreign platforms."}]},   # a paraphrase, not a quote
]}
fake = ScriptedClaude(json_reply(scripted_batch))
parsed = fake.client.messages.parse(
    model=MODEL, max_tokens=2048, system=SUBAGENT_PROMPT, output_format=ClaimBatch,
    messages=[{"role": "user", "content": as_context([FIXTURE_BY_ID["S2"]])}],
).parsed_output
for c in parsed.claims:
    print(c.claim_id, c.statement, "| period:", c.period)
print("excerpt problems:", excerpt_problems(parsed.claims, TEXTS) or "none")
print("output_config.format sent:", fake.requests[0]["output_config"]["format"]["type"])

> **Observe:** the schema guarantees *shape* (every claim has evidence, every field is present), not *truth*. The paraphrased excerpt passes validation and fails the substring check. Section 12.7 removes this whole class of error for documents you supply: with the Citations API, the API extracts the quote itself.

> **Try it:** add a third scripted claim with `"period": "2025"` for a number the source dates to 2024. Which check would catch it? (None of the above: that is a semantic check you would add, for example "the period must appear in the excerpt or be derivable from the publication date".)

In [ ]:
# (live) a real subagent call: claim records from the official statistics document
live = client.messages.parse(
    model=MODEL, max_tokens=2048, system=SUBAGENT_PROMPT, output_format=ClaimBatch,
    messages=[{"role": "user", "content": as_context([FIXTURE_BY_ID["S2"]])}],
)
for c in live.parsed_output.claims:
    print(c.claim_id, "|", c.statement, "| period:", c.period, "| value:", c.value, c.unit)
print("excerpt problems:", excerpt_problems(live.parsed_output.claims, TEXTS) or "none")

### The reference claim set used in sections 12.2–12.6

These are the claim records an ideal pair of subagents would return for S1–S4 (the mini-project produces them through a real pipeline). Each excerpt is copied from the fixture text, which the assertion at the end verifies.

In [ ]:
def ev(source_id: str, excerpt: str, location: str | None = None) -> Evidence:
    return Evidence(source_id=source_id, excerpt=excerpt, location=location)


MARKET = dict(subject="AI-generated music market", metric="market size", unit="USD bn")
DETECT = dict(subject="AI-audio detection classifier (TN-24-07)", unit="%", period=None)
S1_A = "We estimate the global market for AI-generated music at USD 3.2 billion in 2024, up from USD 1.9 billion in 2023."
S2_A = "Revenue reported by registered firms selling AI-generated music totalled USD 2.1 billion in 2024, compared with USD 1.9 billion in 2023."

CLAIMS = [
    Claim(claim_id="C1", statement="The global AI-generated music market was estimated at USD 3.2 billion in 2024.",
          value=3.2, period="2024", evidence=[ev("S1", S1_A, "Executive summary")], **MARKET),
    Claim(claim_id="C2", statement="The global AI-generated music market was estimated at USD 1.9 billion in 2023.",
          value=1.9, period="2023", evidence=[ev("S1", S1_A, "Executive summary")], **MARKET),
    Claim(claim_id="C3", statement="The market is projected to reach USD 7.5 billion by 2027.",
          value=7.5, period="2027", evidence=[ev("S1", "We project the market to reach USD 7.5 billion by 2027.")], **MARKET),
    Claim(claim_id="C4", statement="Revenue reported by registered firms was a preliminary USD 2.1 billion in 2024.",
          value=2.1, period="2024", evidence=[ev("S2", S2_A, "Section 4"),
                                              ev("S2", "Figures for 2024 are preliminary and will be revised in the final release.")], **MARKET),
    Claim(claim_id="C5", statement="Revenue reported by registered firms was USD 1.9 billion in 2023.",
          value=1.9, period="2023", evidence=[ev("S2", S2_A, "Section 4")], **MARKET),
    Claim(claim_id="C6", statement="Three major record labels announced licensing agreements with AI music start-ups.",
          subject="Record-label licensing of AI music", metric="labels announcing deals", value=3, unit="labels",
          period="2024-09", evidence=[ev("S3", "Three major record labels announced licensing agreements with AI music start-ups this week.")]),
    Claim(claim_id="C7", statement="Label executives described the agreements as preliminary.",
          subject="Record-label licensing of AI music", metric=None, value=None, unit=None, period="2024-09",
          evidence=[ev("S3", "Label executives said the agreements were preliminary and could change.")]),
    Claim(claim_id="C8", statement="The classifier reached 94% precision on a 10,000-track benchmark.", metric="precision", value=94,
          evidence=[ev("S4", "Our classifier reached 94% precision and 81% recall on a benchmark of 10,000 tracks.")], **DETECT),
    Claim(claim_id="C9", statement="The classifier reached 81% recall on a 10,000-track benchmark.", metric="recall", value=81,
          evidence=[ev("S4", "Our classifier reached 94% precision and 81% recall on a benchmark of 10,000 tracks.")], **DETECT),
    Claim(claim_id="C10", statement="Recall drops to 62% for tracks mixing human vocals with generated instrumentals.",
          metric="recall (mixed tracks)", value=62,
          evidence=[ev("S4", "Recall drops to 62% for tracks that mix human vocals with generated instrumentals.")], **DETECT),
]
CLAIM_BY_ID = {c.claim_id: c for c in CLAIMS}
assert not excerpt_problems(CLAIMS, TEXTS)
print(len(CLAIMS), "claims, all excerpts verbatim")

## 12.2 Handling conflicting data

When two credible sources disagree, the analysis agent **finishes its work, keeps both values with attribution and annotates the conflict**; the **coordinator** reconciles before synthesis. Before calling it a conflict, normalize units and classify: definition/scope difference, temporal difference, undetermined (a date is missing), methodology difference, or a genuine conflict. [Theory](README.md#122-handling-conflicting-data)

### Normalize, then classify (offline)

`classify()` is the README version plus unit normalization: units are converted to a base unit first (USD 3,200 million and USD 3.2 billion are the same number), so claims about different quantities are still recognized as such. Periods are compared **before** values, so the same value for two different years is a time series, not an agreement.

In [ ]:
UNIT_SCALE = {"usd bn": ("USD bn", 1.0), "usd m": ("USD bn", 0.001), "%": ("%", 1.0), "labels": ("labels", 1.0)}


def normalized(c: Claim) -> tuple[float | None, str | None]:
    if c.value is None or c.unit is None:
        return c.value, c.unit
    base, factor = UNIT_SCALE.get(c.unit.lower(), (c.unit, 1.0))
    return round(c.value * factor, 6), base


def same_quantity(a: Claim, b: Claim) -> bool:
    return (a.subject.lower(), (a.metric or "").lower(), normalized(a)[1]) == \
           (b.subject.lower(), (b.metric or "").lower(), normalized(b)[1])


def classify(a: Claim, b: Claim, sources: dict[str, Source], rel_tol: float = 0.05) -> str:
    if not same_quantity(a, b):
        return "different_quantity"        # e.g. % of uploads vs % of releases: report both, explain
    va, vb = normalized(a)[0], normalized(b)[0]
    if va is None or vb is None:
        return "not_numeric"
    if a.period is None or b.period is None:
        return "undetermined"              # cannot tell a trend from a conflict without dates
    if a.period != b.period:
        return "temporal_difference"       # a time series (even if the value did not change), not a contradiction
    if abs(va - vb) <= rel_tol * max(abs(va), abs(vb)):
        return "agree"                     # same quantity, same period, same value: merge the evidence (12.5)
    methods = {sources[e.source_id].methodology for e in a.evidence + b.evidence}
    if None not in methods and len(methods) > 1:
        return "methodology_difference"    # both valid, measured differently
    return "conflict"                      # same thing, same period, different numbers: escalate


# Two extra fixture sources with the SAME method, to show a genuine conflict
EXTRA = {
    "X1": Source(source_id="X1", title="Label Panel Survey A", url=None, publisher="Example Panel Co.", source_type="industry_report",
                 published="2025-03-01", retrieved_at=None, methodology="survey of record labels", derived_from=None),
    "X2": Source(source_id="X2", title="Label Panel Survey B", url=None, publisher="Other Panel Ltd.", source_type="industry_report",
                 published="2025-04-01", retrieved_at=None, methodology="survey of record labels", derived_from=None),
}
ALL_SOURCES = {**SOURCES, **EXTRA}
c = CLAIM_BY_ID
pairs = {
    "C1 vs C4 (3.2 vs 2.1, both 2024)": (c["C1"], c["C4"]),
    "C2 vs C5 (1.9 vs 1.9, 2023)": (c["C2"], c["C5"]),
    "C2 vs C1 (1.9 in 2023 vs 3.2 in 2024)": (c["C2"], c["C1"]),
    "C2 vs 1.9 in 2024 (same value, new year)": (c["C2"], c["C2"].model_copy(update={"claim_id": "Cz", "period": "2024"})),
    "C1 vs USD 3,200 m": (c["C1"], c["C1"].model_copy(update={"claim_id": "Cx", "value": 3200, "unit": "USD m"})),
    "C1 vs undated 2.6": (c["C1"], c["C1"].model_copy(update={"claim_id": "Cy", "value": 2.6, "period": None})),
    "precision vs recall": (c["C8"], c["C9"]),
    "survey A vs survey B": (
        c["C4"].model_copy(update={"claim_id": "Ca", "value": 2.4, "evidence": [ev("X1", "... USD 2.4 billion in 2024 ...")]}),
        c["C4"].model_copy(update={"claim_id": "Cb", "value": 1.6, "evidence": [ev("X2", "... USD 1.6 billion in 2024 ...")]})),
}
for label, (a, b) in pairs.items():
    print(f"{label:40} -> {classify(a, b, ALL_SOURCES)}")

### Conflict records and coordinator reconciliation (offline)

The analysis agent writes the conflict record and **does not decide**. The coordinator applies a policy per conflict type, can send a verification request (here a stub `verify_fact`), and records *why* it reconciled the way it did.

In [ ]:
def conflict_record(kid: str, claims: list[Claim], sources: dict[str, Source], kind: str) -> dict:
    values = []
    for cl in claims:
        e = cl.evidence[0]
        s = sources[e.source_id]
        values.append({"value": cl.value, "unit": cl.unit, "claim_id": cl.claim_id, "source_id": s.source_id,
                       "source": s.title, "published": s.published, "period": cl.period,
                       "methodology": s.methodology, "excerpt": e.excerpt})
    methods = " vs ".join(v["methodology"] or "method not stated" for v in values)
    return {"conflict_id": kid, "subject": claims[0].subject, "metric": claims[0].metric, "values": values,
            "conflict_detected": True, "conflict_type": kind,
            "possible_explanation": f"Different measurement: {methods}." if kind == "methodology_difference" else None,
            "status": "unresolved", "reconciliation": None}


def reconcile(record: dict, verify: Callable[[dict], dict | None] | None = None) -> dict:
    """Coordinator policy. Never silently overwrite a value: record the decision and the reason."""
    kind, out = record["conflict_type"], dict(record)
    if kind == "methodology_difference":
        out.update(status="explained", reconciliation="Present both values with their methods and scope in 'Contested'; do not average.")
    elif kind == "undetermined":
        out.update(status="needs_dates", reconciliation="Ask the subagent for the data period of each value; do not guess.")
    elif kind == "conflict":
        check = verify(record) if verify else None
        if check:
            out.update(status="verified", reconciliation=f"Independent check: {check['finding']} [{check['source_id']}]. Keep all values visible.")
        else:
            out.update(status="unresolved", reconciliation="Present both in 'Contested'; route to human review if a decision depends on it.")
    return out


def verify_fact(record: dict) -> dict | None:
    """Stub for a scoped verification tool (a third, independent source)."""
    return {"source_id": "X3", "finding": "a third label survey reports USD 2.0 billion for 2024"}


k1 = conflict_record("K1", [c["C1"], c["C4"]], ALL_SOURCES, classify(c["C1"], c["C4"], ALL_SOURCES))
print(json.dumps(reconcile(k1), indent=1)[:1400])
a, b = pairs["survey A vs survey B"]
k2 = conflict_record("K2", [a, b], ALL_SOURCES, classify(a, b, ALL_SOURCES))
print("\nK2:", reconcile(k2, verify=verify_fact)["reconciliation"])

> **Observe:** 3.2 vs 2.1 for the same year is *not* resolved by picking a winner. The record keeps both values, both excerpts, both methods and both publication dates, and the coordinator's decision is written into `reconciliation` with a reason. The 2023 values agree, so they are not a conflict at all: they become one claim with two sources (12.5).

> **Try it:** set S2's `methodology` to `None` and rerun. The pair becomes `conflict` (method unknown is treated as "same"), which is the conservative choice: an unknown method is not an explanation.

## 12.3 Include dates for correct interpretation

Without dates, a temporal difference looks exactly like a contradiction. Keep three dates apart: the **data period** (`Claim.period`, what time the number describes), the **publication date** (`Source.published`) and the **retrieval date** (`Source.retrieved_at`). Resolve relative dates against the publication date, sort by period before comparing, and apply a freshness policy in code. [Theory](README.md#123-include-dates-for-correct-interpretation)

In [ ]:
def period_key(period: str | None) -> tuple:
    """Sortable key for 'YYYY', 'YYYY-MM', 'YYYY-Qn', 'YYYY-Hn'; unknown periods sort last."""
    if not period:
        return (9999, 99)
    m = re.fullmatch(r"(\d{4})(?:-(Q[1-4]|H[12]|\d{2}))?", period)
    if not m:
        return (9999, 98)
    year, part = int(m.group(1)), m.group(2)
    if part is None:
        return (year, 0)
    if part.startswith("Q"):
        return (year, int(part[1]) * 3)
    if part.startswith("H"):
        return (year, int(part[1]) * 6)
    return (year, int(part))


def resolve_relative(phrase: str, published: str) -> str | None:
    """Resolve a relative time phrase against the SOURCE's publication date (never against today)."""
    pub = date.fromisoformat(published if len(published) == 10 else published + "-01")
    q = (pub.month - 1) // 3 + 1
    p = phrase.lower()
    if "this year" in p:
        return str(pub.year)
    if "last year" in p:
        return str(pub.year - 1)
    if "this week" in p or "this month" in p:
        return f"{pub.year}-{pub.month:02d}"
    if "last quarter" in p:
        return f"{pub.year}-Q{q - 1}" if q > 1 else f"{pub.year - 1}-Q4"
    return None   # unknown phrase: leave the period null and keep the phrase in the excerpt


FRESHNESS_DAYS = {"market statistic": 730, "price": 30, "technical": 1095}


def parse_date(s: str | None) -> date | None:
    if not s:
        return None
    for fmt in ("%Y-%m-%d", "%Y-%m", "%Y"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    return None


def is_stale(published: str | None, kind: str, as_of: date = AS_OF) -> bool | None:
    d = parse_date(published)
    return None if d is None else (as_of - d).days > FRESHNESS_DAYS[kind]


print("S3 says 'this week', published 2024-09-02 ->", resolve_relative("this week", "2024-09-02"))
print("'last quarter' in a report published 2025-02-20 ->", resolve_relative("last quarter", "2025-02-20"))
print("NOT", AS_OF.year, "(today) — the model's sense of 'now' is irrelevant here\n")

print(f"{'source':5} {'published':11} {'retrieved':11} {'stale (market stat)?'}")
for s in SOURCES.values():
    print(f"{s.source_id:5} {s.published or 'unknown':11} {s.retrieved_at or '-':11} {is_stale(s.published, 'market statistic')}")

### A trend, not a contradiction (offline)

The README's 18% vs 26% example (12.3): with dates, the comparison describes growth; without them, the honest output is "undetermined", never "contradiction".

In [ ]:
def describe_pair(a: dict, b: dict) -> str:
    if not a.get("period") or not b.get("period"):
        return (f"{a['source']} reports {a['value']}% and {b['source']} reports {b['value']}%, but at least one data period "
                "is unknown, so we cannot tell a trend from a disagreement.")
    first, second = sorted([a, b], key=lambda x: period_key(x["period"]))
    delta = second["value"] - first["value"]
    return (f"{first['source']} (data for {first['period']}, published {first['published']}) reports {first['value']}%; "
            f"{second['source']} (data for {second['period']}, published {second['published']}) reports {second['value']}%. "
            f"This is consistent with a change of {delta:+g} percentage points over time, not a disagreement.")


A = {"source": "Survey A", "value": 18, "period": "2022", "published": "2023-03"}
B = {"source": "Survey B", "value": 26, "period": "2025", "published": "2026-02"}
print(describe_pair(A, B))
print()
print(describe_pair(A, {**B, "period": None}))

market = sorted([cl for cl in CLAIMS if cl.subject == MARKET["subject"]], key=lambda cl: (period_key(cl.period), cl.claim_id))
print("\nmarket claims in period order:", [(cl.period, cl.value, cl.evidence[0].source_id) for cl in market])

### Dates from the platform, and a stale-index diagnosis (offline)

Web search results carry `page_age` as free text (when the page was last updated, which is not necessarily the date of the fact); web fetch results carry an ISO `retrieved_at`. For your own documents, keep version and dates in metadata: it turns "the RAG system started giving wrong answers" into a one-line diagnosis.

In [ ]:
def parse_page_age(page_age: str | None) -> str | None:
    """'April 30, 2025' -> '2025-04-30'. Unparseable or missing -> None (do not guess)."""
    for fmt in ("%B %d, %Y", "%b %d, %Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(page_age or "", fmt).date().isoformat()
        except ValueError:
            pass
    return None


print([parse_page_age(x) for x in ["April 30, 2025", "September 2, 2024", "3 days ago", None]])

# Chunk metadata of a RAG index right after a document refresh
CURRENT_VERSION = {"licensing-policy": 3}
answer_citations = [{"source": "kb://licensing-policy#v2", "indexed_at": "2026-06-01"},
                    {"source": "kb://licensing-policy#v3", "indexed_at": "2026-09-28"}]


def stale_citations(citations: list[dict], current: dict[str, int]) -> list[str]:
    out = []
    for cit in citations:
        m = re.fullmatch(r"kb://([\w-]+)#v(\d+)", cit["source"])
        if m and int(m.group(2)) < current.get(m.group(1), 0):
            out.append(f"{cit['source']} is superseded by v{current[m.group(1)]} (indexed {cit['indexed_at']})")
    return out


print(stale_citations(answer_citations, CURRENT_VERSION))

> **Observe:** "this week" in an article published 2024-09-02 means September 2024, not this week. And the stale-index check points the investigation at retrieval and indexing (an old version is still being retrieved), not at the prompt or the model.

> **Try it:** give the model today's date in a system prompt (`f"Today's date is {AS_OF.isoformat()}."`) and ask it which fixture source is the most recent. Then ask without the date. A model's default sense of "now" is its training cutoff.

## 12.4 Render by content type

Match the form to the content: statistics as a **table** (value, unit, period, source, method on one row), news as **prose** with inline citations, technical findings as a **structured list**, time series in **chronological** order, and conflicts in a **contested** section with both values. Keep the sources' own qualifiers. Let code render tables and references from claim records; let the model write the narrative by claim ID. [Theory](README.md#124-render-by-content-type)

In [ ]:
KIND = {"C1": "statistic", "C2": "statistic", "C3": "forecast", "C4": "statistic", "C5": "statistic",
        "C6": "event", "C7": "event", "C8": "technical", "C9": "technical", "C10": "technical"}
CONTENT_FORMAT = {"statistic": "table", "forecast": "table", "event": "prose", "technical": "list"}
QUALIFIERS = ["estimat", "preliminary", "project", "up to", "about", "approximately", "could", "may "]


def cap(s: str) -> str:
    """Citation blocks are often sentence fragments: capitalize and close them for display."""
    s = s.strip()
    return s[:1].upper() + s[1:] + ("" if s.endswith((".", "!", "?")) else ".")


def fmt_value(cl: Claim) -> str:
    if cl.value is None:
        return "-"
    v = f"{cl.value:g} {cl.unit or ''}".strip()
    q = [w for w in ("estimat", "preliminary", "project") if w in cl.statement.lower()]
    return v + (f" ({'estimate' if q[0] == 'estimat' else 'projected' if q[0] == 'project' else q[0]})" if q else "")


def source_cell(cl: Claim, sources: dict[str, Source], refs: dict[str, int] | None = None) -> str:
    ids = dict.fromkeys(e.source_id for e in cl.evidence if e.source_id in sources)
    return "; ".join(f"{sources[s].title} [{refs[s] if refs else s}] ({sources[s].published or 'date unknown'})" for s in ids)


def method_cell(cl: Claim, sources: dict[str, Source]) -> str:
    return "; ".join(dict.fromkeys(sources[e.source_id].methodology or "not stated" for e in cl.evidence))


def render_table(claims: list[Claim], sources: dict[str, Source], refs: dict[str, int] | None = None) -> str:
    rows = sorted(claims, key=lambda cl: ((cl.metric or ""), period_key(cl.period), cl.claim_id))
    lines = ["| Metric | Value | Period | Source (published) | Method |", "|---|---|---|---|---|"]
    lines += [f"| {cl.metric or '-'} | {fmt_value(cl)} | {cl.period or 'unknown'} | {source_cell(cl, sources, refs)} | {method_cell(cl, sources)} |"
              for cl in rows]
    return "\n".join(lines)


def render_list(claims: list[Claim]) -> str:
    return "\n".join(f"- **{cl.statement}** Evidence: \"{cl.evidence[0].excerpt}\" [{cl.evidence[0].source_id}]" for cl in claims)


def render_timeline(claims: list[Claim]) -> str:
    return "\n".join(f"- **{cl.period}**: {fmt_value(cl)} [{', '.join(dict.fromkeys(e.source_id for e in cl.evidence))}]"
                     for cl in sorted(claims, key=lambda cl: (period_key(cl.period), cl.claim_id)))


def dropped_qualifiers(statement: str, excerpt: str) -> list[str]:
    return [q.strip() for q in QUALIFIERS if q in excerpt.lower() and q not in statement.lower()]


by_format = defaultdict(list)
for cl in CLAIMS:
    by_format[CONTENT_FORMAT[KIND[cl.claim_id]]].append(cl)
display(Markdown("**Key figures (table)**\n\n" + render_table(by_format["table"], SOURCES)))
display(Markdown("**Market size over time (timeline)**\n\n" + render_timeline([cl for cl in CLAIMS if cl.subject == MARKET["subject"]])))
display(Markdown("**Technical findings (list)**\n\n" + render_list(by_format["list"])))

### Prose with inline citations, written by the model by claim ID (offline)

The model writes sentences that end in claim IDs; code turns IDs into numbered references. The numbers in the table never pass through the model, so they cannot drift.

In [ ]:
def reference_numbers(claims: list[Claim]) -> dict[str, int]:
    used = sorted({e.source_id for cl in claims for e in cl.evidence}, key=lambda s: (len(s), s))
    return {sid: i for i, sid in enumerate(used, 1)}


def link_refs(sentence: str, claims_by_id: dict[str, Claim], refs: dict[str, int]) -> str:
    def repl(m: re.Match) -> str:
        cl = claims_by_id[m.group(1)]
        return "".join(f"[{refs[sid]}]" for sid in dict.fromkeys(e.source_id for e in cl.evidence))
    return re.sub(r"\[(C\d+)\]", repl, sentence)


REFS = reference_numbers(CLAIMS)
draft = ("Three major record labels announced licensing agreements with AI music start-ups in September 2024 [C6], "
         "and executives described the agreements as preliminary [C7].")
print(link_refs(draft, CLAIM_BY_ID, REFS))
print("references:", {n: SOURCES[s].title for s, n in REFS.items()})

print("\nqualifier check:")
for statement, excerpt in [("AI tracks are 30% of uploads.", "AI tracks are up to 30% of uploads."),
                           (c["C4"].statement, c["C4"].evidence[1].excerpt + " " + c["C4"].evidence[0].excerpt),
                           ("The market will reach USD 7.5 billion by 2027.", c["C3"].evidence[0].excerpt)]:
    print(f"  {statement[:60]!r:64} dropped: {dropped_qualifiers(statement, excerpt) or 'none'}")

> **Observe:** "up to 30%" became "30%" and "We project ... will reach" became a plain "will reach": `dropped_qualifiers()` flags both. The preliminary official figure keeps its hedge, so it passes.

> **Try it:** render the market claims as prose ("The market was 1.9 in 2023, 3.2 or 2.1 in 2024, ...") and compare it with the table above. Which one lets a reader see in one glance that 2024 is contested and why?

## 12.5 Carrying mappings through every hop

Four rules: **pass by reference** (forward claim records or a pointer to them, never a retelling), **merge, never replace** (same subject, metric, unit, period and value: union the evidence), **check independence** (count origins, not URLs), and make the synthesis output **reference claim IDs** so code can audit it. [Theory](README.md#125-carrying-mappings-through-every-hop-expanded-beyond-the-guide)

In [ ]:
def key(c: Claim) -> tuple:
    return (c.subject.lower(), (c.metric or "").lower(), normalized(c)[1] or "", c.period)


def merge_claims(claims: list[Claim]) -> list[Claim]:
    """Union the evidence of claims that state the same value for the same thing; keep the rest apart."""
    merged: dict[tuple, Claim] = {}
    for cl in claims:
        k = key(cl) + (normalized(cl)[0],)
        if k in merged:
            seen = {(e.source_id, e.excerpt) for e in merged[k].evidence}
            merged[k].evidence += [e for e in cl.evidence if (e.source_id, e.excerpt) not in seen]
        else:
            merged[k] = cl.model_copy(deep=True)
    return list(merged.values())


def independent_origins(c: Claim, sources: dict[str, Source]) -> int:
    return len({sources[e.source_id].derived_from or e.source_id for e in c.evidence})


merged = merge_claims(CLAIMS)
print(f"{len(CLAIMS)} claims -> {len(merged)} after merge (inputs untouched: C2 still has {len(CLAIM_BY_ID['C2'].evidence)} evidence item)")
m2023 = next(m for m in merged if m.period == "2023")
print("2023 market:", m2023.claim_id, "evidence from", [e.source_id for e in m2023.evidence],
      "| independent origins:", independent_origins(m2023, SOURCES))

# Syndication: three articles that all re-report the same press release
PR = Source(source_id="PR1", title="AI Music Inc. press release", url="https://aimusic.example/press", publisher="AI Music Inc.",
            source_type="company_filing", published="2025-01-10", retrieved_at=None, methodology=None, derived_from=None)
syndicated = {f"N{i}": Source(source_id=f"N{i}", title=f"Article {i}", url=f"https://news{i}.example.com/a", publisher=f"Outlet {i}",
                              source_type="news", published="2025-01-11", retrieved_at=None, methodology=None, derived_from="PR1")
              for i in (1, 2, 3)}
REG = {"PR1": PR, **syndicated}
claim = Claim(claim_id="C99", statement="AI Music Inc. says it has 2 million users.", subject="AI Music Inc.", metric="users",
              value=2, unit="million", period="2025-01",
              evidence=[ev(f"N{i}", "AI Music Inc. says it now has 2 million users.") for i in (1, 2, 3)])
print("\nURLs:", len(claim.evidence), "| independent origins:", independent_origins(claim, REG))
no_lineage = {k: v.model_copy(update={"derived_from": None}) for k, v in REG.items()}
print("without derived_from recorded:", independent_origins(claim, no_lineage), "<- looks 'well established', and is not")

### Pass by reference, not by retelling (offline)

The coordinator stores each subagent's batch and forwards a **reference**; the synthesis step loads the records. Compare what reaches synthesis when the coordinator instead "helpfully" retells the findings.

In [ ]:
class ClaimStore:
    """A tiny file-backed store: subagents write batches, the coordinator passes refs (paths), synthesis reads."""

    def __init__(self, root: Path | None = None):
        self.root = Path(root or tempfile.mkdtemp(prefix="claims-"))

    def put(self, name: str, claims: list[Claim]) -> str:
        path = self.root / f"{name}.json"
        path.write_text(json.dumps([cl.model_dump() for cl in claims], indent=1))
        return str(path)

    def get(self, ref: str, model: type[Claim] = Claim) -> list[Claim]:
        return [model.model_validate(x) for x in json.loads(Path(ref).read_text())]


store = ClaimStore()
ref = store.put("doc_analysis", CLAIMS)
by_reference = store.get(ref)

fake = ScriptedClaude(reply(text("The subagents found that the AI music market is about USD 3.2 billion and growing fast, "
                                 "labels are signing deals, and detectors are about 81% accurate.")))
retold = fake.client.messages.create(model=MODEL, max_tokens=300, messages=[{"role": "user", "content":
    "Summarize these findings for the synthesis agent:\n" + json.dumps([cl.statement for cl in CLAIMS])}]).content[0].text

print("by reference: ", len(by_reference), "claims,", sum(len(cl.evidence) for cl in by_reference), "evidence items, ref =", Path(ref).name)
print("retold:       ", attribution_report(retold), "|", retold[:90])

### The Agent SDK hop: validate the subagent's final text (offline)

On an Agent SDK subagent hop there are no API citation blocks: the subagent's **final text** is what returns to the coordinator, so the schema in the subagent's prompt is what carries provenance. The coordinator code validates it with Pydantic and turns a bad batch into structured error context instead of passing prose along. This notebook does not start the SDK; a definition would look like this:

```python
from claude_agent_sdk import AgentDefinition, ClaudeAgentOptions

options = ClaudeAgentOptions(
    allowed_tools=["Agent", "Read"],          # "Agent" was called "Task" in older SDKs and in the exam guide
    agents={"doc-analyst": AgentDefinition(
        description="Extracts claim records from documents",
        prompt=SUBAGENT_PROMPT + "\nReturn ONLY a JSON object {\"claims\": [...]} matching the claim schema.",
        tools=["Read"])},
)
```

In [ ]:
def parse_subagent_output(final_text: str) -> dict:
    """Coordinator side: accept claim records, or return structured error context (never prose)."""
    m = re.search(r"```(?:json)?\s*(\{.*\})\s*```", final_text, re.S) or re.search(r"(\{.*\})", final_text, re.S)
    if not m:
        return {"status": "failed", "error_type": "no_json", "detail": final_text[:120]}
    try:
        return {"status": "ok", "claims": ClaimBatch.model_validate_json(m.group(1)).claims}
    except ValidationError as exc:
        return {"status": "failed", "error_type": "schema_violation",
                "detail": [f"{'.'.join(map(str, err['loc']))}: {err['msg']}" for err in exc.errors()][:3]}


good = "Here are my findings:\n```json\n" + ClaimBatch(claims=CLAIMS[:2]).model_dump_json() + "\n```"
no_evidence = '{"claims": [{"claim_id": "C1", "statement": "The market is USD 3.2 billion.", "subject": "AI music", ' \
              '"metric": "market size", "value": 3.2, "unit": "USD bn", "period": "2024", "evidence": []}]}'
for label, txt in [("good", good), ("no evidence", no_evidence), ("prose", "The market is big and growing.")]:
    out = parse_subagent_output(txt)
    print(f"{label:12} -> {out['status']}", out.get("error_type", ""), out.get("detail", f"{len(out.get('claims', []))} claims"))

> **Observe:** merging turned the two 2023 claims into one claim with two independent sources, while three syndicated articles stay **one** origin. And the retold summary reaching synthesis has numbers but no sources and no hedges: the same loss as in 12.1, now inside the coordinator.

> **Try it:** change `merge_claims` to *replace* instead of union (`merged[k] = cl`). Rerun and look at how many sources the 2023 claim has. That one-line bug is the "merge, never replace" rule.

## 12.6 Coverage and uncertainty reporting

Track coverage **per sub-question** as data with four states: `covered`, `empty` (searched, no matches: a valid answer), `failed` (could not look: a gap) and `not_searched`. Derive support levels from evidence (independent origins, freshness, conflicts), not from the model's self-rated confidence. [Theory](README.md#126-coverage-and-uncertainty-reporting-expanded-beyond-the-guide)

In [ ]:
class CoverageItem(BaseModel):
    sub_question: str
    source_category: str
    state: Literal["covered", "empty", "failed", "not_searched"]
    detail: str | None
    claim_ids: list[str]


def coverage_from_web_search(block, sub_question: str, category: str) -> CoverageItem:
    """Map a web_search_tool_result block: error object -> failed, empty list -> empty, results -> covered."""
    if not isinstance(block.content, list):
        return CoverageItem(sub_question=sub_question, source_category=category, state="failed",
                            detail=f"web search error: {block.content.error_code}", claim_ids=[])
    if not block.content:
        return CoverageItem(sub_question=sub_question, source_category=category, state="empty",
                            detail="search succeeded with 0 results", claim_ids=[])
    return CoverageItem(sub_question=sub_question, source_category=category, state="covered",
                        detail=f"{len(block.content)} results", claim_ids=[])


# Three recorded web_search_tool_result blocks, parsed by the real SDK
fake = ScriptedClaude(reply(
    {"type": "web_search_tool_result", "tool_use_id": "srvtoolu_1", "content": [
        {"type": "web_search_result", "url": "https://news.example.com/labels-ai-licensing", "title": "Major labels sign...",
         "encrypted_content": "enc-1", "page_age": "September 2, 2024"}]},
    {"type": "web_search_tool_result", "tool_use_id": "srvtoolu_2", "content": []},
    {"type": "web_search_tool_result", "tool_use_id": "srvtoolu_3",
     "content": {"type": "web_search_tool_result_error", "error_code": "unavailable"}},
))
blocks = fake.client.messages.create(model=MODEL, max_tokens=10, messages=[{"role": "user", "content": "x"}]).content
for blk, (q, cat) in zip(blocks, [("Q3", "news"), ("Q6", "regulation news"), ("Q5", "social media")]):
    item = coverage_from_web_search(blk, q, cat)
    print(f"{type(blk).__name__:26} -> {item.state:8} ({item.detail})")

In [ ]:
def support_level(cl: Claim, sources: dict[str, Source], contested_ids: set[str], kind: str = "market statistic",
                  as_of: date = AS_OF) -> str:
    if cl.claim_id in contested_ids:
        return "contested"
    fresh = [is_stale(sources[e.source_id].published, kind, as_of) is False for e in cl.evidence]
    origins = independent_origins(cl, sources)
    if origins >= 2:
        return "well_established" if any(fresh) else "corroborated_but_dated"
    return "single_source"


def render_coverage(items: list[CoverageItem], questions: dict[str, str], single_source: list[str]) -> str:
    lines = ["## Coverage and limitations"]
    for it in items:
        q = questions[it.sub_question]
        if it.state == "covered":
            lines.append(f"- {it.sub_question} ({q}): covered by {it.source_category} [{', '.join(it.claim_ids)}].")
        elif it.state == "empty":
            lines.append(f"- {it.sub_question} ({q}): no sources found in {it.source_category} ({it.detail}).")
        elif it.state == "failed":
            lines.append(f"- {it.sub_question} ({q}): {it.source_category} were unavailable ({it.detail}); "
                         "findings on this question may be incomplete.")
        else:
            lines.append(f"- {it.sub_question} ({q}): not examined ({it.detail or 'out of scope'}).")
    if single_source:
        lines.append(f"- Single-source findings: {', '.join(single_source)}.")
    return "\n".join(lines)


contested = {"C1", "C4"}   # from conflict K1
for cl in merge_claims(CLAIMS):
    print(f"{cl.claim_id:4} {support_level(cl, SOURCES, contested):17} {cl.statement[:70]}")

QUESTIONS = {"Q1": "market size", "Q5": "listener reception", "Q6": "regulation", "Q7": "non-English sources"}
items = [CoverageItem(sub_question="Q1", source_category="industry and official reports", state="covered", detail=None, claim_ids=["C1", "C2", "C4"]),
         CoverageItem(sub_question="Q5", source_category="news archives and social feeds", state="failed", detail="timeout after 2 attempts", claim_ids=[]),
         CoverageItem(sub_question="Q6", source_category="web news", state="empty", detail="search succeeded with 0 results", claim_ids=[]),
         CoverageItem(sub_question="Q7", source_category="-", state="not_searched", detail="out of scope for this brief", claim_ids=[])]
display(Markdown(render_coverage(items, QUESTIONS, single_source=["C3", "C6"])))
print("A single number would say:", f"{sum(i.state == 'covered' for i in items) / len(items):.0%} coverage  <- which question is missing? why?")

> **Observe:** the 2023 market figure is the only well-established finding (two independent origins, one fresh enough under the 730-day policy); the 2024 values are contested; everything else rests on one source and says so. `empty` and `failed` produce different sentences: "no sources found" is an answer, "were unavailable" is a gap.

> **Try it:** call `support_level(..., as_of=date(2027, 6, 1))` for the merged 2023 claim. Both sources are now older than the 730-day policy, so it becomes `corroborated_but_dated`. Freshness is a policy decision you make per domain, in code.

## 12.7 The Citations API: document blocks

Send documents with `"citations": {"enabled": true}` and Claude's answer comes back as text blocks whose `citations` point at exact locations: `char_location` (plain text, 0-indexed, end exclusive), `page_location` (PDF, **1-indexed**, end exclusive) or `content_block_location` (custom content, 0-indexed, end exclusive). `cited_text` is extracted by the API and does not count toward output tokens. Citations are all-or-none across documents in a request and **cannot be combined with structured outputs** (400). [Theory](README.md#127-the-citations-api-document-blocks-expanded-beyond-the-guide)

### Build the request, with a pre-flight guard (offline)

In [ ]:
def document_block(doc: dict, citations: bool = True) -> dict:
    """A plain-text document; metadata goes in `context` (passed to the model, never cited)."""
    return {
        "type": "document",
        "source": {"type": "text", "media_type": "text/plain", "data": doc["text"]},
        "title": doc["title"],
        "context": json.dumps({"source_id": doc["source_id"], "published": doc["published"], "url": doc["url"]}),
        "citations": {"enabled": citations},
    }


def custom_content_block(title: str, chunks: list[str], context: str) -> dict:
    """Custom content: your blocks are the citable units, no further chunking."""
    return {"type": "document", "source": {"type": "content", "content": [{"type": "text", "text": t} for t in chunks]},
            "title": title, "context": context, "citations": {"enabled": True}}


def pdf_block(base64_data: str, title: str) -> dict:
    return {"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": base64_data},
            "title": title, "citations": {"enabled": True}}


def citation_request_problems(request: dict) -> list[str]:
    """Predict the 400s before paying for a request."""
    flags = []
    for m in request.get("messages", []):
        if isinstance(m.get("content"), list):
            for b in m["content"]:
                blocks = b.get("content", []) if b.get("type") == "tool_result" and isinstance(b.get("content"), list) else [b]
                for x in blocks:
                    if x.get("type") in ("document", "search_result"):
                        flags.append(bool((x.get("citations") or {}).get("enabled")))
    problems = []
    if any(flags) and not all(flags):
        problems.append("citations must be enabled on all document/search_result blocks or on none")
    if any(flags) and (request.get("output_config", {}).get("format") or request.get("output_format")):
        problems.append("citations + structured outputs (output_config.format) returns a 400: use the two-step design")
    return problems


QUESTION = ("How large is the AI-generated music market, and how reliable is AI-music detection? "
            "Write one short sentence per figure and keep each source's qualifiers.")
cited_docs = [FIXTURE_BY_ID[s] for s in ("S1", "S2", "S4")]
request = {"model": MODEL, "max_tokens": 1024,
           "messages": [{"role": "user", "content": [document_block(d) for d in cited_docs] + [{"type": "text", "text": QUESTION}]}]}
print("ok request:", citation_request_problems(request) or "no problems")

mixed = json.loads(json.dumps(request))
mixed["messages"][0]["content"][1]["citations"]["enabled"] = False
print("mixed:     ", citation_request_problems(mixed))
with_schema = {**request, "output_config": {"format": {"type": "json_schema", "schema": {"type": "object"}}}}
print("+ schema:  ", citation_request_problems(with_schema))

### Convert a recorded cited response into claim records (offline)

The recorded response below has the exact shape the API returns: text split into blocks, sourced blocks carrying `char_location` citations. This is the converter 12.7 describes, for all five citation types, and it maps `document_index` back to *your* `source_id` (the model never retypes a URL or a quote). The integrity check slices the original text with the returned indices.

In [ ]:
def locate(cit) -> str | None:
    match cit.type:
        case "char_location":
            return f"chars {cit.start_char_index}-{cit.end_char_index}"
        case "page_location":
            last = cit.end_page_number - 1                       # end is exclusive
            return f"p. {cit.start_page_number}" if last <= cit.start_page_number else f"pp. {cit.start_page_number}-{last}"
        case "content_block_location" | "search_result_location":
            last = cit.end_block_index - 1                       # end is exclusive
            return f"block {last}" if last == cit.start_block_index else f"blocks {cit.start_block_index}-{last}"
        case "web_search_result_location":
            return None                                          # the URL is the location
    return None


def resolve_document(cit, docs: list[dict]) -> dict | None:
    """Map a citation's document_index to your registry entry, checked against document_title.

    The docs define document_index as 0-indexed over all document blocks in the request, across all messages.
    They do not say how pages that web fetch inserts are numbered relative to your own document blocks,
    so never trust the index alone: confirm the title, fall back to a unique title match, else give up loudly."""
    i, title = cit.document_index, getattr(cit, "document_title", None)
    if 0 <= i < len(docs) and (title is None or docs[i].get("title") in (None, title)):
        return docs[i]
    matches = [d for d in docs if title is not None and d.get("title") == title]
    return matches[0] if len(matches) == 1 else None


def evidence_from_citation(cit, docs: list[dict], url_ids: dict[str, str] | None = None) -> dict:
    if cit.type in ("char_location", "page_location", "content_block_location"):
        doc = resolve_document(cit, docs)                      # index over ALL document blocks, title-checked
        source_id = doc["source_id"] if doc else f"UNRESOLVED:doc{cit.document_index}"
    elif cit.type == "search_result_location":
        source_id = cit.source
    else:                                                        # web_search_result_location
        source_id = (url_ids or {}).get(cit.url, cit.url)
    return {"source_id": source_id, "excerpt": cit.cited_text, "location": locate(cit)}


def claims_from_response(response, docs: list[dict], url_ids: dict[str, str] | None = None,
                         prefix: str = "C") -> tuple[list[dict], list[str]]:
    claims, unsupported = [], []
    for block in (b for b in response.content if b.type == "text"):
        if not block.citations:
            if re.search(r"\d", block.text):                     # a number with no citation: flag it
                unsupported.append(block.text.strip())
            continue
        claims.append({"claim_id": f"{prefix}{len(claims) + 1}", "statement": block.text.strip(" ,.;"),
                       "evidence": [evidence_from_citation(c, docs, url_ids) for c in block.citations]})
    return claims, unsupported


def verify_char_citations(response, docs: list[dict]) -> list[str]:
    problems = []
    for b in response.content:
        for c in (getattr(b, "citations", None) or []):
            if c.type == "char_location":
                doc = resolve_document(c, docs)
                if doc is None or "text" not in doc:
                    problems.append(f"doc {c.document_index} ({c.document_title!r}) not in the registry: cannot verify")
                    continue
                actual = doc["text"][c.start_char_index:c.end_char_index]
                if actual.strip() != c.cited_text.strip():
                    problems.append(f"doc {c.document_index} [{c.start_char_index}:{c.end_char_index}] != cited_text")
    return problems


S1_B = "We project the market to reach USD 7.5 billion by 2027."
S2_C = "Figures for 2024 are preliminary and will be revised in the final release."
S4_A = "Our classifier reached 94% precision and 81% recall on a benchmark of 10,000 tracks."
S4_B = "Recall drops to 62% for tracks that mix human vocals with generated instrumentals."
S4_D = "We recommend human review for any track scored between 0.4 and 0.6."
D = cited_docs
DOC_REPLY = reply(
    text("According to the analyst report, "),
    text("the global AI-generated music market was estimated at USD 3.2 billion in 2024", [char_cite(D, 0, S1_A)]),
    text("; "),
    text("the 2023 market was put at USD 1.9 billion", [char_cite(D, 0, S1_A)]),
    text(" (roughly a 68% increase since). "),
    text("The analyst report projects the market to reach USD 7.5 billion by 2027", [char_cite(D, 0, S1_B)]),
    text(". Official statistics give a lower figure: "),
    text("revenue reported by registered firms totalled a preliminary USD 2.1 billion in 2024", [char_cite(D, 1, S2_A), char_cite(D, 1, S2_C)]),
    text(". "),
    text("Registered firms reported USD 1.9 billion for 2023", [char_cite(D, 1, S2_A)]),
    text(". On detection, "),
    text("a classifier reached 94% precision on a benchmark of 10,000 tracks", [char_cite(D, 2, S4_A)]),
    text(", and "),
    text("the classifier reached 81% recall on the same benchmark", [char_cite(D, 2, S4_A)]),
    text("; "),
    text("recall drops to 62% for tracks that mix human vocals with generated instrumentals", [char_cite(D, 2, S4_B)]),
    text(". "),
    text("The authors recommend human review for tracks scored between 0.4 and 0.6", [char_cite(D, 2, S4_D)]),
    text("."),
)

fake = ScriptedClaude(DOC_REPLY)
response = fake.client.messages.create(**request)
show(response, width=70)
drafts, unsupported = claims_from_response(response, cited_docs, prefix="D")
print()
for d in drafts[:3]:
    print(json.dumps(d)[:200])
print("...", len(drafts), "cited claims | unsupported numeric text:", unsupported)
print("integrity check:", verify_char_citations(response, cited_docs) or "all cited_text matches the source slice")

> **Observe:** the SDK parsed each citation into a real `CitationCharLocation` object. The model's own arithmetic ("roughly a 68% increase") arrived **without** a citation, so the converter flags it instead of letting a computed number masquerade as a sourced one. Note `document_index` counts *all* document blocks in the request, across all messages: keep your registry in the same order as the blocks. `resolve_document` also checks `document_title` against the registry, so a numbering surprise (for example pages inserted by web fetch, 12.9) becomes a visible `UNRESOLVED` source instead of a claim silently attached to the wrong document.

> **Try it:** change `start_char_index` of one recorded citation by 5 and rerun the integrity check. In production that check is cheap insurance on every response.

### PDF pages and custom content (offline)

The same converter handles `page_location` (1-indexed pages, end exclusive) and `content_block_location` (your own blocks). Custom content is how you control granularity, for example one block per transcript turn.

In [ ]:
TRANSCRIPT = ["Analyst: Our 2024 estimate is USD 3.2 billion.",
              "Regulator: Our preliminary count is USD 2.1 billion.",
              "Analyst: The gap is mostly foreign platforms."]
docs2 = [{"source_id": "S1-PDF", "title": "Global AI Music Market Report 2024 (PDF)"},
         {"source_id": "PANEL-7", "title": "Panel transcript, 2025-03-12"}]
fake = ScriptedClaude(reply(
    text("The full report states the 2024 estimate on pages 3 and 4", [page_cite(0, docs2[0]["title"], S1_A, 3, 5)]),
    text("; in a panel, the analyst attributed the gap to foreign platforms",
         [block_cite(1, docs2[1]["title"], TRANSCRIPT, 2, 3)]),
    text("."),
))
r = fake.client.messages.create(model=MODEL, max_tokens=512, messages=[{"role": "user", "content": [
    pdf_block("JVBERi0xLjQK...", docs2[0]["title"]),                       # placeholder bytes; shape only
    custom_content_block(docs2[1]["title"], TRANSCRIPT, json.dumps({"source_id": "PANEL-7", "date": "2025-03-12"})),
    {"type": "text", "text": "What explains the gap between the two 2024 figures?"}]}])
for d in claims_from_response(r, docs2)[0]:
    print(d["evidence"][0]["source_id"], "|", d["evidence"][0]["location"], "|", d["statement"])
print("custom content block sent:", json.dumps(fake.requests[0]["messages"][0]["content"][1]["source"])[:110], "...")

### Streaming: citations arrive as `citations_delta` (offline)

When streaming, each citation arrives as its own `citations_delta` inside `content_block_delta`, and the SDK's stream helper emits a `citation` event and accumulates it onto the text block.

In [ ]:
def sse(*events: tuple[str, dict]) -> str:
    return "".join(f"event: {name}\ndata: {json.dumps(data)}\n\n" for name, data in events)


cit = char_cite(cited_docs, 1, S2_A)
STREAM = sse(
    ("message_start", {"type": "message_start", "message": {"id": "msg_s", "type": "message", "role": "assistant", "model": "scripted",
                       "content": [], "stop_reason": None, "stop_sequence": None, "usage": {"input_tokens": 1, "output_tokens": 0}}}),
    ("content_block_start", {"type": "content_block_start", "index": 0, "content_block": {"type": "text", "text": "", "citations": []}}),
    ("content_block_delta", {"type": "content_block_delta", "index": 0, "delta": {"type": "text_delta", "text": "Registered firms reported USD 2.1 billion in 2024"}}),
    ("content_block_delta", {"type": "content_block_delta", "index": 0, "delta": {"type": "citations_delta", "citation": cit}}),
    ("content_block_stop", {"type": "content_block_stop", "index": 0}),
    ("message_delta", {"type": "message_delta", "delta": {"stop_reason": "end_turn", "stop_sequence": None}, "usage": {"output_tokens": 12}}),
    ("message_stop", {"type": "message_stop"}),
)
fake = ScriptedClaude(STREAM)
with fake.client.messages.stream(**request) as stream:
    for event in stream:
        if event.type == "citation":
            print("citation event ->", event.citation.type, repr(event.citation.cited_text[:50]))
    final = stream.get_final_message()
print("accumulated on the block:", [c.type for c in final.content[0].citations])

### The two-step design: cited answer, then structured fields (offline)

Citations and structured outputs cannot share a request, so step 2 is a *separate* structured call over the already-cited claims. It fills `subject`, `metric`, `value`, `unit`, `period`, while `claim_id` and `evidence` from step 1 stay untouched.

In [ ]:
class ClaimFields(BaseModel):
    claim_id: str
    subject: str
    metric: str | None
    value: float | None
    unit: str | None
    period: str | None
    kind: Literal["statistic", "forecast", "event", "technical"]
    sub_question: Literal["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7"]


class ClaimFieldsBatch(BaseModel):
    items: list[ClaimFields]


class ReportClaim(Claim):
    kind: Literal["statistic", "forecast", "event", "technical"]
    sub_question: str


FIELDS_PROMPT = """For each cited claim, fill subject, metric, value, unit, period, kind and sub_question.
Use only the statement and its excerpts. period = the time the value describes; resolve relative phrases
("this week", "this year") against the source's publication date shown in brackets. Use null when not stated.
Units: "USD bn" for billions of US dollars, "%" for percentages. Use the same subject and metric wording
for claims about the same quantity. Never change claim_id.
Sub-questions: Q1 market size, Q2 growth outlook, Q3 industry response, Q4 detection reliability,
Q5 listener reception, Q6 regulation, Q7 non-English sources."""


def enrich(llm: anthropic.Anthropic, drafts: list[dict], sources: dict[str, Source]) -> list[ReportClaim]:
    payload = [{"claim_id": d["claim_id"], "statement": d["statement"],
                "excerpts": [f"[{e['source_id']}, published {sources[e['source_id']].published}] {e['excerpt']}" for e in d["evidence"]]}
               for d in drafts]
    fields = llm.messages.parse(model=MODEL, max_tokens=4096, system=FIELDS_PROMPT, output_format=ClaimFieldsBatch,
                                messages=[{"role": "user", "content": json.dumps(payload, indent=1)}]).parsed_output
    by_id = {f.claim_id: f for f in fields.items}
    out = []
    for d in drafts:
        f = by_id.get(d["claim_id"])
        if f is None:
            continue                                           # the coordinator logs it as un-enriched
        out.append(ReportClaim(claim_id=d["claim_id"], statement=d["statement"], evidence=[Evidence(**e) for e in d["evidence"]],
                               **f.model_dump(exclude={"claim_id"})))
    return out


def fields(cid, subject, metric, value, unit, period, kind, q):
    return {"claim_id": cid, "subject": subject, "metric": metric, "value": value, "unit": unit, "period": period,
            "kind": kind, "sub_question": q}


M, T = "AI-generated music market", "AI-audio detection classifier (TN-24-07)"
DOC_FIELDS = json_reply({"items": [
    fields("D1", M, "market size", 3.2, "USD bn", "2024", "statistic", "Q1"),
    fields("D2", M, "market size", 1.9, "USD bn", "2023", "statistic", "Q1"),
    fields("D3", M, "market size", 7.5, "USD bn", "2027", "forecast", "Q2"),
    fields("D4", M, "market size", 2.1, "USD bn", "2024", "statistic", "Q1"),
    fields("D5", M, "market size", 1.9, "USD bn", "2023", "statistic", "Q1"),
    fields("D6", T, "precision", 94, "%", None, "technical", "Q4"),
    fields("D7", T, "recall", 81, "%", None, "technical", "Q4"),
    fields("D8", T, "recall (mixed tracks)", 62, "%", None, "technical", "Q4"),
    fields("D9", T, "human review score band", None, None, None, "technical", "Q4"),
]})
fake = ScriptedClaude(DOC_FIELDS)
enriched = enrich(fake.client, drafts, SOURCES)
for cl in enriched[:4]:
    print(cl.claim_id, cl.kind, cl.value, cl.unit, cl.period, "| evidence:", [(e.source_id, e.location) for e in cl.evidence])
print("step-2 request has citations?", "citations" in json.dumps(fake.requests[0]["messages"]), "| has output_config:", "output_config" in fake.requests[0])

In [ ]:
# (live) a real cited answer over S1, S2 and S4, converted to claim records in code
live = client.messages.create(**request)
show(live, width=70)
live_drafts, live_unsupported = claims_from_response(live, cited_docs)
print("\ncited claims:", len(live_drafts), "| unsupported numeric text:", live_unsupported)
print("integrity:", verify_char_citations(live, cited_docs) or "OK")

## 12.8 `search_result` blocks: citations for your own retrieval

Return your RAG chunks as `search_result` blocks (`source`, `title`, `content` = text blocks, `citations.enabled`) and Claude cites them with `search_result_location` (your `source` and `title`, block indices, `cited_text`). The text block is the smallest citable unit, so **chunking sets citation granularity**. In a `tool_result`, if one block is a `search_result`, all must be; citations are all-or-nothing across search results. [Theory](README.md#128-search_result-blocks-citations-for-your-own-retrieval-expanded-beyond-the-guide)

In [ ]:
KB = [
    {"doc_id": "licensing-policy", "version": 3, "title": "AI licensing policy", "published": "2025-11-04",
     "paragraphs": ["Catalogue licences for AI training require written approval from the rights committee.",
                    "Royalty rates for AI training licences are reviewed every six months.",
                    "Licences never cover vocal likeness; voice cloning needs separate artist consent."]},
    {"doc_id": "detection-runbook", "version": 1, "title": "AI-track detection runbook", "published": "2025-06-30",
     "paragraphs": ["Tracks scored between 0.4 and 0.6 go to a human reviewer.",
                    "Reviewers record the decision and the evidence in the moderation log."]},
]


def retriever_search(query: str, k: int = 2) -> list[dict]:
    """Keyword-overlap retriever (stand-in for vector/hybrid search)."""
    words = set(re.findall(r"\w+", query.lower()))
    scored = [(len(words & set(re.findall(r"\w+", " ".join(d["paragraphs"]).lower()))), d) for d in KB]
    return [d for s, d in sorted(scored, key=lambda x: -x[0]) if s > 0][:k]


def kb_search(query: str, granularity: Literal["paragraph", "document"] = "paragraph") -> list[dict]:
    hits = retriever_search(query)
    if not hits:
        return [{"type": "text", "text": "No results found."}]          # empty is a valid answer
    return [{"type": "search_result",
             "source": f"kb://{h['doc_id']}#v{h['version']}",              # stable ID, with version
             "title": f"{h['title']} (published {h['published']})",
             "content": ([{"type": "text", "text": p} for p in h["paragraphs"]] if granularity == "paragraph"
                         else [{"type": "text", "text": " ".join(h["paragraphs"])}]),
             "citations": {"enabled": True}}
            for h in hits]


def tool_result_problems(content: list[dict]) -> list[str]:
    types = {b["type"] for b in content}
    problems = []
    if "search_result" in types and types != {"search_result"}:
        problems.append("a tool_result with any search_result must contain only search_result blocks; "
                        "put supporting text inside a search result's content")
    flags = {bool((b.get("citations") or {}).get("enabled")) for b in content if b["type"] == "search_result"}
    if len(flags) > 1:
        problems.append("citations must be enabled on all search results or on none")
    if any(not t.get("text") for b in content if b["type"] == "search_result" for t in b["content"]):
        problems.append("search_result content blocks must be non-empty text")
    return problems


KB_TOOL = {"name": "kb_search",
           "description": ("Search the internal knowledge base of licensing policies and moderation runbooks. Returns the "
                           "matching documents with their version and publication date. Use it for questions about internal "
                           "policy or procedure. Do not use it for market statistics or news."),
           "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"],
                            "additionalProperties": False}}

results = kb_search("royalty rates for AI training licences")
print("tool_result content OK?", tool_result_problems(results) or "yes", "|", [r["source"] for r in results])
print("with a stray note:     ", tool_result_problems(results + [{"type": "text", "text": "Note: results may be stale."}]))

In [ ]:
def kb_agent(llm: anthropic.Anthropic, question: str, max_turns: int = 4):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_turns):
        r = llm.messages.create(model=MODEL, max_tokens=1024, tools=[KB_TOOL], messages=messages)
        messages.append({"role": "assistant", "content": r.content})
        if r.stop_reason != "tool_use":
            return r, messages
        results = []
        for b in r.content:
            if b.type == "tool_use":
                content = kb_search(b.input["query"])
                assert not tool_result_problems(content)
                results.append({"type": "tool_result", "tool_use_id": b.id, "content": content})
        messages.append({"role": "user", "content": results})
    raise RuntimeError("max_turns reached")


q = "How often are royalty rates for AI training licences reviewed, and do licences cover voice cloning?"
first_results = kb_search("royalty rates AI training licences voice cloning")
policy = first_results[0]
fake = ScriptedClaude(
    reply(tool_call("kb_search", {"query": "royalty rates AI training licences voice cloning"}, id="toolu_kb1")),
    reply(text("Royalty rates are reviewed every six months", [sr_cite(0, policy, 1, 2)]),
          text(", and "),
          text("licences never cover vocal likeness, so voice cloning needs separate artist consent", [sr_cite(0, policy, 2, 3)]),
          text(".")),
)
final, history = kb_agent(fake.client, q)
show(final)
print("\nwhat we sent back:", json.dumps(fake.requests[1]["messages"][-1]["content"][0]["content"][0])[:160], "...")
print("as evidence:", [evidence_from_citation(c, []) for b in final.content if b.type == "text" for c in (b.citations or [])][0])

### Granularity is a chunking decision (offline)

In [ ]:
para = kb_search("royalty rates", "paragraph")[0]
whole = kb_search("royalty rates", "document")[0]
print("paragraph blocks -> cited_text:", repr(sr_cite(0, para, 1, 2)["cited_text"]))
print("one big block    -> cited_text:", repr(sr_cite(0, whole, 0, 1)["cited_text"][:140] + "..."))

> **Observe:** the citation carries *your* `source` string (`kb://licensing-policy#v3`), so the version travels with the claim, which is exactly what the stale-index diagnosis in 12.3 needs. With one block per document, every citation returns the whole document: Claude cites whole blocks, never substrings.

> **Try it:** make `kb_search` return a `search_result` plus a plain `text` block in the same `tool_result` and run `tool_result_problems()`. The real API rejects that request; the fix is to move the note inside a search result's `content`, where it also becomes citable.

In [ ]:
# (live) the same KB agent with the real model
final, history = kb_agent(client, q)
show(final)

## 12.9 Web search and web fetch citations

Web search citations are **always on** (`web_search_result_location`: `url`, `title`, `encrypted_index`, `cited_text` up to 150 characters) and each result has `page_age`. Web fetch citations are **optional** (`"citations": {"enabled": true}` on the tool); the fetched page arrives as a `document` with a `retrieved_at` timestamp. Send assistant blocks back unchanged (including `encrypted_content` / `encrypted_index`), and when you show API output to end users, show its citations. [Theory](README.md#129-web-search-and-web-fetch-citations-expanded-beyond-the-guide)

In [ ]:
WEB_TOOLS = [
    {"type": "web_search_20250305", "name": "web_search", "max_uses": 3,
     "allowed_domains": ["news.example.com", "stats.example.gov"]},          # allowed_domains OR blocked_domains, not both
    {"type": "web_fetch_20250910", "name": "web_fetch", "max_uses": 2, "max_content_tokens": 20000,
     "citations": {"enabled": True}},
]


def web_tool_problems(tools: list[dict]) -> list[str]:
    return [f"{t['name']}: use allowed_domains or blocked_domains, not both" for t in tools
            if t.get("allowed_domains") and t.get("blocked_domains")]


print(web_tool_problems(WEB_TOOLS) or "tools OK",
      "|", web_tool_problems([{**WEB_TOOLS[0], "blocked_domains": ["spam.example"]}]))

S3 = FIXTURE_BY_ID["S3"]
S3_A = "Three major record labels announced licensing agreements with AI music start-ups this week."
S3_B = "Label executives said the agreements were preliminary and could change."
fetched_doc = [{"source_id": "S3", "title": S3["title"], "text": S3["text"]}]
WEB_REPLY = reply(
    {"type": "server_tool_use", "id": "srvtoolu_w1", "name": "web_search", "input": {"query": "record labels AI music licensing deals 2024"}},
    {"type": "web_search_tool_result", "tool_use_id": "srvtoolu_w1", "content": [
        {"type": "web_search_result", "url": S3["url"], "title": S3["title"], "encrypted_content": "enc-w1-0", "page_age": "September 2, 2024"},
        {"type": "web_search_result", "url": "https://blog.example.net/ai-music-opinion", "title": "Why AI music is overhyped",
         "encrypted_content": "enc-w1-1", "page_age": None}]},
    {"type": "server_tool_use", "id": "srvtoolu_w2", "name": "web_search", "input": {"query": "AI-generated music regulation royalties 2024"}},
    {"type": "web_search_tool_result", "tool_use_id": "srvtoolu_w2", "content": []},
    {"type": "server_tool_use", "id": "srvtoolu_f1", "name": "web_fetch", "input": {"url": S3["url"]}},
    {"type": "web_fetch_tool_result", "tool_use_id": "srvtoolu_f1", "content": {
        "type": "web_fetch_result", "url": S3["url"], "retrieved_at": "2026-10-01T09:30:00Z",
        "content": {"type": "document", "source": {"type": "text", "media_type": "text/plain", "data": S3["text"]},
                    "title": S3["title"], "citations": {"enabled": True}}}},
    text("Three major record labels announced licensing agreements with AI music start-ups in early September 2024",
         [web_cite(S3["url"], S3["title"], S3_A)]),
    text("; "),
    text("executives described the agreements as preliminary", [char_cite(fetched_doc, 0, S3_B)]),
    text(". I found no reporting on regulation of AI music royalties."),
)
fake = ScriptedClaude(WEB_REPLY)
web_msgs = [{"role": "user", "content": "How are record labels responding to AI music, and is regulation coming? Cite sources."}]
web_response = fake.client.messages.create(model=MODEL, max_tokens=2048, tools=WEB_TOOLS, messages=web_msgs)
show(web_response, width=70)

### Sources, dates and claims from a web response (offline)

In [ ]:
def web_sources(response) -> dict[str, dict]:
    """url -> {title, page_age, published (parsed), retrieved_at} from search and fetch result blocks."""
    out: dict[str, dict] = {}
    for b in response.content:
        if b.type == "web_search_tool_result" and isinstance(b.content, list):
            for r in b.content:
                out.setdefault(r.url, {"title": r.title, "page_age": r.page_age, "published": parse_page_age(r.page_age),
                                       "retrieved_at": None})
        elif b.type == "web_fetch_tool_result" and b.content.type == "web_fetch_result":
            out.setdefault(b.content.url, {"title": b.content.content.title, "page_age": None, "published": None})
            out[b.content.url]["retrieved_at"] = b.content.retrieved_at
    return out


def search_log(response) -> list[dict]:
    queries = {b.id: b.input.get("query") for b in response.content if b.type == "server_tool_use" and b.name == "web_search"}
    log = []
    for b in response.content:
        if b.type == "web_search_tool_result":
            if isinstance(b.content, list):
                log.append({"query": queries.get(b.tool_use_id), "state": "covered" if b.content else "empty", "n": len(b.content)})
            else:
                log.append({"query": queries.get(b.tool_use_id), "state": "failed", "error": b.content.error_code})
    return log


def fetched_docs(response) -> list[dict]:
    """Fetched pages arrive as document blocks; list them in order with their titles.

    How these pages are numbered in document_index next to your own document blocks is not documented,
    so evidence_from_citation resolves them by index *and* title (resolve_document) rather than by index alone."""
    return [{"source_id": b.content.url, "title": b.content.content.title}
            for b in response.content if b.type == "web_fetch_tool_result" and b.content.type == "web_fetch_result"]


ws = web_sources(web_response)
for url, meta in ws.items():
    print(url, "|", meta)
print("\nsearch log:", search_log(web_response))
url_ids = {S3["url"]: "S3"}
web_drafts, web_unsupported = claims_from_response(web_response, [{**d, "source_id": url_ids.get(d["source_id"], d["source_id"])}
                                                                  for d in fetched_docs(web_response)], url_ids, prefix="W")
for d in web_drafts:
    print(d["claim_id"], d["evidence"][0], "|", d["statement"])

### Your documents and fetched pages in one request (offline)

The Citations docs define `document_index` over *all document blocks in the request*, but they do not say how a page that web fetch inserts is numbered next to the `document` blocks you sent yourself. Do not build on a guess. Resolve every document citation by index **and** `document_title`, and turn anything that does not match into an explicit `UNRESOLVED` source that the audit (12.10) will block. The recorded citation below deliberately uses an index that points at the wrong registry entry.

In [ ]:
own_and_fetched = [{"source_id": "S1", "title": FIXTURE_BY_ID["S1"]["title"], "text": FIXTURE_BY_ID["S1"]["text"]},
                   {"source_id": "S3", "title": S3["title"], "text": S3["text"]}]
wrong_index = {**char_cite(fetched_doc, 0, S3_B), "document_index": 0}       # index 0 is S1 in our registry
unknown = {**char_cite(fetched_doc, 0, S3_B), "document_index": 7, "document_title": "Some page we never registered"}
probe = ScriptedClaude(reply(text("executives described the agreements as preliminary", [wrong_index]),
                             text("; another claim", [unknown])))
pr = probe.client.messages.create(model=MODEL, max_tokens=256, messages=[{"role": "user", "content": "x"}])
for b in pr.content:
    for c in b.citations or []:
        naive = own_and_fetched[c.document_index]["source_id"] if c.document_index < len(own_and_fetched) else "IndexError"
        print(f"index {c.document_index} -> naive: {naive:10} | title-checked: {evidence_from_citation(c, own_and_fetched)['source_id']}")
print("integrity:", verify_char_citations(pr, own_and_fetched))

> **Observe:** the naive index lookup attaches the quote to S1, a document that never said it. The title check maps it to S3, and a citation that matches nothing becomes `UNRESOLVED:doc7`, which no audit will accept as evidence. The simplest way to avoid the question entirely is to keep web fetch and your own `document` blocks in **separate calls** when you need exact mappings.

### Display obligation, multi-turn, and `pause_turn` (offline)

Render the answer *with* its citations; send the assistant content back unchanged on the next turn (the encrypted fields must survive); and continue a paused server-side loop by resending the paused assistant message.

In [ ]:
def render_with_citations(response) -> str:
    """Inline [n] markers plus a source list: what an end user should see."""
    urls, parts = [], []
    for b in response.content:
        if b.type != "text":
            continue
        parts.append(b.text)
        for c in b.citations or []:
            target = c.url if c.type == "web_search_result_location" else (c.document_title or "document")
            if target not in urls:
                urls.append(target)
            parts.append(f"[{urls.index(target) + 1}]")
    return "".join(parts) + "\n\n" + "\n".join(f"{i}. {u}" for i, u in enumerate(urls, 1))


print(render_with_citations(web_response))

follow_up = ScriptedClaude(reply(text("No regulation found so far.")))
follow_up.client.messages.create(model=MODEL, max_tokens=512, tools=WEB_TOOLS, messages=web_msgs + [
    {"role": "assistant", "content": web_response.content},            # unchanged, encrypted fields included
    {"role": "user", "content": "Which of these deals are final?"}])
sent = json.dumps(follow_up.requests[0]["messages"][1])
print("\nencrypted_content kept:", "enc-w1-0" in sent, "| encrypted_index kept:", "enc-idx-" in sent)


def run_server_tools(llm: anthropic.Anthropic, messages: list, tools: list[dict], max_continuations: int = 3):
    for _ in range(max_continuations + 1):
        r = llm.messages.create(model=MODEL, max_tokens=2048, tools=tools, messages=messages)
        if r.stop_reason != "pause_turn":
            return r
        messages = messages + [{"role": "assistant", "content": r.content}]   # resend unchanged to continue
    raise RuntimeError("server-side loop did not finish")


paused = ScriptedClaude(reply({"type": "server_tool_use", "id": "srvtoolu_p", "name": "web_search", "input": {"query": "q"}},
                              stop_reason="pause_turn"), WEB_REPLY)
final = run_server_tools(paused.client, web_msgs, WEB_TOOLS)
print("after pause_turn:", final.stop_reason, "| requests sent:", len(paused.requests))

> **Observe:** web search gave a URL, a 150-character-max excerpt and a free-text `page_age`; web fetch gave the full page plus `retrieved_at`, so the second claim cites an exact character range. The empty regulation search lands in the search log as `empty`, which the mini-project turns into a coverage line. The source list shows the fetched page twice (its search URL and its fetched title); a production renderer should de-duplicate by source, for example by mapping each fetched document back to its URL.

> **Try it:** delete `"encrypted_content"` from one result in the follow-up request (edit the dict before sending) and think about what the real API does: it returns a 400, because it decrypts that content to restore the search results in Claude's context.

In [ ]:
# (live, estimated ~$0.02-0.05: up to 2 searches at $10 per 1,000 plus result tokens) real web search with citations
live_web = client.messages.create(
    model=MODEL, max_tokens=2048,
    tools=[{"type": "web_search_20250305", "name": "web_search", "max_uses": 2}],
    messages=[{"role": "user", "content": "What is the latest generally available Claude model? Answer in two sentences and cite sources."}],
)
show(live_web, width=80)
print(render_with_citations(live_web))
print({u: m["page_age"] for u, m in web_sources(live_web).items()})

## 12.10 Verifying provenance before you publish

Citations make claims *checkable*; something still has to *check* them. Run deterministic checks in code first (every sentence has claim IDs, referenced claims exist, numbers appear in the evidence, contested claims are never stated one-sided, gaps are mentioned), then a fresh-instance model review where the stakes justify it, then humans for legal, financial, regulatory or decision-driving contested findings. [Theory](README.md#1210-verifying-provenance-before-you-publish-expanded-beyond-the-guide)

In [ ]:
CLAIM_REF = re.compile(r"\[(C\d+)\]")
NUMBER = re.compile(r"\d+(?:[.,]\d+)*")


def norm_num(s: str) -> str:
    return s.replace(",", "")


def allowed_numbers(cl: Claim, sources: dict[str, Source]) -> set[str]:
    """Number tokens a sentence may use: the excerpts plus the provenance metadata (period, publication dates)."""
    parts = [e.excerpt for e in cl.evidence] + [cl.period or ""] + [sources[e.source_id].published or "" for e in cl.evidence]
    return {norm_num(n) for n in NUMBER.findall(" ".join(parts))}


def audit(sections: dict[str, list[str]], claims: dict[str, Claim], sources: dict[str, Source],
          conflicts: list[dict], coverage: list[CoverageItem], report_md: str | None = None) -> list[str]:
    problems = []
    conflict_sets = [{v["claim_id"] for v in k["values"]} for k in conflicts]
    for section, sentences in sections.items():
        for s in sentences:
            refs = CLAIM_REF.findall(s)
            if not refs:
                problems.append(f"[{section}] no claim reference: {s[:70]}")
                continue
            missing = [r for r in refs if r not in claims]
            if missing:
                problems.append(f"[{section}] unknown claims {missing}: {s[:70]}")
                continue
            allowed = set().union(*(allowed_numbers(claims[r], sources) for r in refs))
            for num in NUMBER.findall(CLAIM_REF.sub("", s)):
                if norm_num(num) not in allowed:   # whole-token match: "20" is not in "2024"
                    problems.append(f"[{section}] number {num} not in evidence of {refs}: {s[:70]}")
            for group in conflict_sets:
                if group & set(refs) and not group <= set(refs):
                    problems.append(f"[{section}] states one side of a conflict {sorted(group)} only: {s[:70]}")
    if report_md is not None:
        for item in coverage:
            if item.state in ("failed", "empty") and item.sub_question not in report_md:
                problems.append(f"coverage gap {item.sub_question} ({item.state}) is not mentioned in the report")
    return problems


k1 = conflict_record("K1", [CLAIM_BY_ID["C1"], CLAIM_BY_ID["C4"]], SOURCES, "methodology_difference")
draft = {"summary": [
    "The AI-generated music market reached USD 3.2 billion in 2024 [C1].",
    "Both sources put the 2023 market at USD 1.9 billion [C2][C5].",
    "The market will double again by 2026.",
    "Detection reaches 91% recall [C9].",
]}
for p in audit(draft, CLAIM_BY_ID, SOURCES, [k1], items):
    print("✗", p)

### Model-side checks: retract, review with a fresh instance, verify narrowly (offline)

Anthropic's hallucination guidance: have Claude find a supporting quote for each claim and **retract** what it cannot support; let it say "I don't have enough information". For an independent check, use a **fresh** instance that sees only the draft and the evidence. For frequent simple checks during synthesis, give the synthesis agent a narrowly scoped `verify_fact` tool; complex investigations still go through the coordinator.

In [ ]:
RETRACT_PROMPT = """Below is a draft and the evidence it may use. For each sentence, find a direct quote in the evidence
that supports it. If you cannot find one, remove the sentence and write [RETRACTED: reason] in its place.
Do not add new information. If the evidence is insufficient overall, say "I don't have enough information"."""


def fresh_review_request(draft_sentences: list[str], claims: dict[str, Claim]) -> dict:
    """A NEW conversation: the reviewer never sees the drafting conversation, only the draft and the evidence."""
    evidence = {cid: [e.model_dump() for e in cl.evidence] for cid, cl in claims.items()}
    return {"model": MODEL, "max_tokens": 1024, "system": RETRACT_PROMPT,
            "messages": [{"role": "user", "content": f"<draft>\n{chr(10).join(draft_sentences)}\n</draft>\n"
                                                     f"<evidence>\n{json.dumps(evidence)}\n</evidence>"}]}


VERIFY_FACT_TOOL = {
    "name": "verify_fact",
    "description": ("Check ONE short factual statement against the claim store and return the matching claim IDs and excerpts, "
                    "or 'not found'. Use it for quick checks while writing (a number, a date, a name). "
                    "Do not use it for open research questions; ask the coordinator for those."),
    "input_schema": {"type": "object", "properties": {"statement": {"type": "string"}}, "required": ["statement"],
                     "additionalProperties": False},
}


def verify_fact_handler(statement: str, claims: dict[str, Claim] = CLAIM_BY_ID) -> dict:
    nums = {norm_num(n) for n in NUMBER.findall(statement)}
    hits = [cid for cid, cl in claims.items() if nums and nums <= allowed_numbers(cl, SOURCES)]
    exact = [cid for cid in hits if claims[cid].value is not None and f"{claims[cid].value:g}" in nums
             and (claims[cid].period is None or claims[cid].period in nums)]    # prefer claims ABOUT this value
    hits = exact or hits
    return {"matches": [{"claim_id": h, "excerpt": claims[h].evidence[0].excerpt} for h in hits]} if hits else {"matches": "not found"}


req = fresh_review_request(draft["summary"], {k: CLAIM_BY_ID[k] for k in ("C1", "C2", "C5", "C9")})
fake = ScriptedClaude(reply(text("1. Keep, but add the 2.1 billion official figure.\n2. Supported.\n"
                                 "3. [RETRACTED: no evidence for 2026]\n4. [RETRACTED: evidence says 81%, not 91%]")))
print(fake.client.messages.create(**req).content[0].text)
print("\nreviewer saw", len(req["messages"]), "message (fresh context)")
print("verify_fact('USD 1.9 billion in 2023') ->", [m["claim_id"] for m in verify_fact_handler("USD 1.9 billion in 2023")["matches"]])
print("verify_fact('91% recall') ->", verify_fact_handler("91% recall"))


def needs_human_review(cl: Claim, level: str, domain: str, decision_relevant: bool) -> bool:
    return domain in {"legal", "medical", "financial", "regulatory"} or (level == "contested" and decision_relevant) \
        or (level == "single_source" and decision_relevant)


print("\nhuman review for C1 (contested, drives a budget decision)?", needs_human_review(CLAIM_BY_ID["C1"], "contested", "market", True))

> **Observe:** the code audit caught all three problems in the draft (a one-sided conflict, an unsupported forecast with no claim reference, and a number that does not appear in its evidence) with no model call; the correctly sourced 2023 sentence passed. The reviewer is a separate request with no history; a reviewer inside the drafting conversation tends to approve its own work.

> **Try it:** write a sentence that cites the right claim but overstates it ("Detection is reliable [C9]."). The code audit passes it; only a reviewer (model or human) can catch overreach. That is why the checks are layered.

## Mini-project: a multi-source research synthesizer

Everything above in one pipeline, coordinator in the middle (hub and spoke):

- **Subagents in parallel:** *doc-analysis* reads S1, S2, S4 with the **Citations API** (12.7) and runs the **two-step** enrichment; *web-research* uses web search + fetch citations (12.9) and logs empty searches; *archive* times out twice and returns **structured error context** instead of failing the run.
- **Coordinator:** merges claims (12.5), classifies pairs and writes conflict records, reconciles them (12.2), derives support levels and coverage (12.6), stores claims **by reference**.
- **Synthesis:** the model writes the summary and the developments prose **by claim ID**; code renders the key-figures table, timeline, technical list, contested section, coverage and numbered references (12.4).
- **Audit:** the code audit (12.10) blocks publication and sends the exact problems back for one revision.

Offline it runs on recorded responses; the live path at the end runs the doc-analysis subagent and synthesis against the real API.

In [ ]:
SUB_QUESTIONS = {"Q1": "How large is the AI-generated music market?", "Q2": "How is it expected to grow?",
                 "Q3": "How are record labels responding?", "Q4": "How reliable is AI-music detection?",
                 "Q5": "How do listeners react (news archives, social media)?", "Q6": "Are regulators acting on AI music royalties?",
                 "Q7": "What do non-English sources say?"}
ASSIGNED = {"Q1": ["doc_analysis"], "Q2": ["doc_analysis"], "Q3": ["web_research"], "Q4": ["doc_analysis"],
            "Q5": ["archive"], "Q6": ["web_research"], "Q7": []}
CATEGORY = {"doc_analysis": "industry, official and technical reports", "web_research": "web news",
            "archive": "news archives and social feeds"}
DOC_SYSTEM = (f"Today's date is {AS_OF.isoformat()}. Answer only from the documents. Write one short sentence per figure, "
              "keep each source's qualifiers (estimated, preliminary, projected) and do not compute new numbers.")


def doc_analysis_subagent(llm: anthropic.Anthropic, docs: list[dict]) -> dict:
    req = {"model": MODEL, "max_tokens": 2048, "system": DOC_SYSTEM,
           "messages": [{"role": "user", "content": [document_block(d) for d in docs] + [{"type": "text", "text": QUESTION}]}]}
    assert not citation_request_problems(req)
    response = llm.messages.create(**req)                                   # step 1: cited answer
    drafts, unsupported = claims_from_response(response, docs, prefix="D")
    claims = enrich(llm, drafts, SOURCES)                                    # step 2: structured fields, evidence untouched
    return {"agent": "doc_analysis", "status": "ok", "claims": claims, "unsupported": unsupported,
            "integrity": verify_char_citations(response, docs)}


def web_research_subagent(llm: anthropic.Anthropic) -> dict:
    response = llm.messages.create(model=MODEL, max_tokens=2048, tools=WEB_TOOLS, messages=web_msgs)
    url_ids = {S3["url"]: "S3"}                                              # known URLs map to the registry
    for url, meta in web_sources(response).items():                          # unknown URLs get registered, dates not guessed
        if url not in url_ids and url not in SOURCES:
            SOURCES[url] = Source(source_id=url, title=meta["title"], url=url, publisher=None, source_type="other",
                                  published=meta["published"], retrieved_at=meta.get("retrieved_at"), methodology=None, derived_from=None)
    docs = [{**d, "source_id": url_ids.get(d["source_id"], d["source_id"])} for d in fetched_docs(response)]
    drafts, unsupported = claims_from_response(response, docs, url_ids, prefix="W")
    return {"agent": "web_research", "status": "ok", "claims": enrich(llm, drafts, SOURCES),   # step 2 on the same client
            "unsupported": unsupported, "search_log": search_log(response)}


def archive_subagent(fetch: Callable[[], list], attempts: int = 2) -> dict:
    last = None
    for _ in range(attempts):
        try:
            return {"agent": "archive", "status": "ok", "claims": fetch()}
        except TimeoutError as exc:
            last = exc
    return {"agent": "archive", "status": "failed", "claims": [], "partial_results": [],
            "error": {"type": "timeout", "source_category": CATEGORY["archive"], "attempts": attempts, "message": str(last)}}


def flaky_archive() -> list:
    raise TimeoutError("archive.example.org did not answer within 10 s")


W = "Record-label licensing of AI music"
WEB_FIELDS = json_reply({"items": [fields("W1", W, "labels announcing deals", 3, "labels", "2024-09", "event", "Q3"),
                                   fields("W2", W, None, None, None, "2024-09", "event", "Q3")]})
print("subagents defined")

In [ ]:
KIND_ORDER = {"statistic": 0, "forecast": 1, "event": 2, "technical": 3}


def coordinate(reports: dict[str, dict], store: ClaimStore) -> dict:
    claims = [cl for r in reports.values() for cl in r["claims"]]
    merged = merge_claims(claims)
    merged.sort(key=lambda cl: (cl.sub_question, KIND_ORDER[cl.kind], period_key(cl.period)))
    renamed = [cl.model_copy(update={"claim_id": f"C{i}"}) for i, cl in enumerate(merged, 1)]
    by_id = {cl.claim_id: cl for cl in renamed}

    conflicts, trends = [], []
    numeric = [cl for cl in renamed if cl.value is not None and cl.kind == "statistic"]
    for i, a in enumerate(numeric):
        for b in numeric[i + 1:]:
            kind = classify(a, b, SOURCES)
            if kind in ("methodology_difference", "conflict", "undetermined"):
                conflicts.append(reconcile(conflict_record(f"K{len(conflicts) + 1}", [a, b], SOURCES, kind)))
            elif kind == "temporal_difference":
                trends.append((a.claim_id, b.claim_id))
    contested = {v["claim_id"] for k in conflicts for v in k["values"]}

    coverage = []
    for q, agents in ASSIGNED.items():
        ids = [cl.claim_id for cl in renamed if cl.sub_question == q]
        if not agents:
            coverage.append(CoverageItem(sub_question=q, source_category="-", state="not_searched", detail="out of scope for this brief", claim_ids=[]))
        elif ids:
            coverage.append(CoverageItem(sub_question=q, source_category=CATEGORY[agents[0]], state="covered", detail=None, claim_ids=ids))
        elif any(reports[a]["status"] == "failed" for a in agents):
            err = next(reports[a]["error"] for a in agents if reports[a]["status"] == "failed")
            coverage.append(CoverageItem(sub_question=q, source_category=err["source_category"], state="failed",
                                         detail=f"{err['type']} after {err['attempts']} attempts", claim_ids=[]))
        else:
            empties = [s["query"] for a in agents for s in reports[a].get("search_log", []) if s["state"] == "empty"]
            coverage.append(CoverageItem(sub_question=q, source_category=CATEGORY[agents[0]], state="empty",
                                         detail=f"searched: {empties[0]!r}, 0 results" if empties else "no matching findings", claim_ids=[]))
    levels = {cid: support_level(cl, SOURCES, contested) for cid, cl in by_id.items()}
    return {"ref": store.put("research_claims", renamed), "conflicts": conflicts, "trends": trends,
            "coverage": coverage, "levels": levels,
            "unsupported": [u for r in reports.values() for u in r.get("unsupported", [])]}


def run_subagents(doc_llm: anthropic.Anthropic, web_llm: anthropic.Anthropic) -> dict[str, dict]:
    with ThreadPoolExecutor(max_workers=3) as pool:       # parallel, like several subagent calls in one turn
        futures = {"doc_analysis": pool.submit(doc_analysis_subagent, doc_llm, cited_docs),
                   "web_research": pool.submit(web_research_subagent, web_llm),
                   "archive": pool.submit(archive_subagent, flaky_archive)}
        return {name: f.result() for name, f in futures.items()}


store = ClaimStore()
reports = run_subagents(ScriptedClaude(DOC_REPLY, DOC_FIELDS).client, ScriptedClaude(WEB_REPLY, WEB_FIELDS).client)
for name, r in reports.items():
    print(f"{name:13} {r['status']:7} claims={len(r['claims'])}", r.get("error", ""))
state = coordinate(reports, store)
CL = {cl.claim_id: cl for cl in store.get(state["ref"], ReportClaim)}
for cid, cl in CL.items():
    print(f"  {cid:4} {cl.sub_question} {cl.kind:9} {state['levels'][cid]:17} {[e.source_id for e in cl.evidence]} {cl.statement[:55]}")
print("conflicts:", [(k["conflict_id"], k["conflict_type"], [v["claim_id"] for v in k["values"]], k["status"]) for k in state["conflicts"]])
print("coverage:", [(c.sub_question, c.state) for c in state["coverage"]])
print("unsupported text from subagents:", state["unsupported"])

In [ ]:
class Narrative(BaseModel):
    summary: list[str]
    developments: list[str]


SYNTH_SYSTEM = """You write the narrative parts of a research brief from claim records.
- Every sentence ends with the IDs of the claims it relies on, e.g. [C2][C3]. Use only the claims given.
- If a claim is contested, state both values with both claim IDs in the same sentence.
- Keep qualifiers (estimated, preliminary, projected). Do not compute or add numbers.
- summary: 3-4 sentences on the main findings. developments: 1-2 sentences on industry news."""


def synthesize(llm: anthropic.Anthropic, ref: str, state: dict, problems: list[str] | None = None,
               previous: Narrative | None = None) -> Narrative:
    claims = store.get(ref, ReportClaim)                                     # loaded by reference
    payload = {"claims": [{"id": cl.claim_id, "statement": cl.statement, "support": state["levels"][cl.claim_id],
                           "kind": cl.kind} for cl in claims],
               "conflicts": [{"id": k["conflict_id"], "claims": [v["claim_id"] for v in k["values"]],
                              "reconciliation": k["reconciliation"]} for k in state["conflicts"]]}
    messages = [{"role": "user", "content": json.dumps(payload, indent=1)}]
    if problems:
        messages += [{"role": "assistant", "content": previous.model_dump_json()},
                     {"role": "user", "content": "The provenance audit failed:\n- " + "\n- ".join(problems)
                      + "\nRevise only the affected sentences. Remove any statement you cannot support with a claim ID."}]
    return llm.messages.parse(model=MODEL, max_tokens=1500, system=SYNTH_SYSTEM, messages=messages,
                              output_format=Narrative).parsed_output


def write_with_audit(llm: anthropic.Anthropic, state: dict, max_revisions: int = 2) -> tuple[Narrative, list[str]]:
    claims = {cl.claim_id: cl for cl in store.get(state["ref"], ReportClaim)}
    narrative = synthesize(llm, state["ref"], state)
    for attempt in range(max_revisions + 1):
        problems = audit(narrative.model_dump(), claims, SOURCES, state["conflicts"], state["coverage"])
        print(f"audit after draft {attempt + 1}: {len(problems)} problem(s)", *[f"\n   - {p}" for p in problems])
        if not problems or attempt == max_revisions:
            return narrative, problems
        narrative = synthesize(llm, state["ref"], state, problems, narrative)

In [ ]:
def render_report(narrative: Narrative, state: dict, problems: list[str]) -> str:
    claims = {cl.claim_id: cl for cl in store.get(state["ref"], ReportClaim)}
    refs = reference_numbers(list(claims.values()))
    link = lambda s: link_refs(s, claims, refs)
    levels = state["levels"]
    lines = lambda items: "\n".join(items) if items else "- (none)"
    parts = [f"# Research brief: AI-generated music (as of {AS_OF.isoformat()})"]
    if problems:
        parts.append("> **Blocked by the provenance audit; needs human review before publishing.**")
    parts += ["## Summary", " ".join(link(s) for s in narrative.summary)]
    est = [cl for cl in claims.values() if levels[cl.claim_id] == "well_established"]
    parts += ["## Well-established findings", lines([f"- {cap(cl.statement)} {link(f'[{cl.claim_id}]')}" for cl in est])]
    parts.append("## Contested findings")
    for k in state["conflicts"]:
        rows = [f"- {v['value']:g} {v['unit']} for {v['period']}: {v['source']} [{refs[v['source_id']]}], published "
                f"{v['published'] or 'date unknown'}, method: {v['methodology'] or 'not stated'}. Excerpt: \"{v['excerpt']}\""
                for v in k["values"]]
        rows.append(f"- Possible explanation: {k['possible_explanation'] or 'not determined'} Coordinator: {k['reconciliation']}")
        parts += [f"**{k['subject']}, {k['metric']} ({k['conflict_type'].replace('_', ' ')})**", lines(rows)]
    figures = [cl for cl in claims.values() if cl.kind in ("statistic", "forecast")]
    parts += ["## Key figures", render_table(figures, SOURCES, refs)]
    parts += ["## Market size over time", lines([f"- **{cl.period}**: {fmt_value(cl)} {link(f'[{cl.claim_id}]')}"
                                                 + (" (contested)" if levels[cl.claim_id] == "contested" else "")
                                                 for cl in sorted(figures, key=lambda c: period_key(c.period))])]
    parts += ["## Developments", " ".join(link(s) for s in narrative.developments)]
    parts += ["## Technical findings", lines([f"- **{cap(cl.statement)}** Evidence: \"{cl.evidence[0].excerpt}\" {link(f'[{cl.claim_id}]')}"
                                              for cl in claims.values() if cl.kind == "technical"])]
    single = [cid for cid, lvl in levels.items() if lvl == "single_source"]
    parts.append(render_coverage(state["coverage"], SUB_QUESTIONS, single))
    sources = [f"{n}. {SOURCES[sid].title}. {SOURCES[sid].publisher or 'publisher unknown'}, published "
               f"{SOURCES[sid].published or 'date unknown'}. {SOURCES[sid].url or 'internal document'} "
               f"(retrieved {SOURCES[sid].retrieved_at or 'n/a'})" for sid, n in sorted(refs.items(), key=lambda x: x[1])]
    parts += ["## Sources", "\n".join(sources)]
    return "\n\n".join(parts)


C = lambda *ids: "".join(f"[{i}]" for i in ids)
DRAFT_1 = {"summary": [f"The AI-generated music market reached USD 3.2 billion in 2024 {C('C2')}.",
                       f"Both the analyst report and official statistics put the 2023 market at USD 1.9 billion {C('C1')}.",
                       "The market will double again by 2026."],
           "developments": [f"Three major record labels announced licensing agreements with AI music start-ups in September 2024 {C('C5')}, "
                            f"and executives described the agreements as preliminary {C('C6')}."]}
DRAFT_2 = {"summary": [f"Estimates of the 2024 market differ: an analyst model puts it at USD 3.2 billion {C('C2')}, while preliminary "
                       f"official statistics count USD 2.1 billion {C('C3')}.",
                       f"Both sources put the 2023 market at USD 1.9 billion {C('C1')}.",
                       f"The analyst report projects USD 7.5 billion by 2027 {C('C4')}."],
           "developments": DRAFT_1["developments"]}
synth_llm = ScriptedClaude(json_reply(DRAFT_1), json_reply(DRAFT_2))
narrative, problems = write_with_audit(synth_llm.client, state)
report = render_report(narrative, state, problems)
final_check = audit(narrative.model_dump(), CL, SOURCES, state["conflicts"], state["coverage"], report_md=report)
print("final audit incl. coverage mentions:", final_check or "pass")
display(Markdown(report))

> **Observe:**
> - The 2023 figure is the only **well-established** finding: two independent sources agree. The 2024 values sit side by side in **Contested** with methods, dates, excerpts and the coordinator's reconciliation; nobody averaged them.
> - The model's own arithmetic ("roughly a 68% increase") was flagged as unsupported at step 1 and never became a claim.
> - The first draft stated one side of the conflict and an invented forecast; the audit blocked it with exact problems and the revision passed. The archive timeout and the empty regulation search appear as **different** lines under Coverage.
> - The key-figures table and the references come from claim records, so their numbers cannot drift during writing.

> **Try it:**
> 1. Make `flaky_archive` succeed and return one claim for Q5; watch the coverage line change from `failed` to `covered`.
> 2. Script a `DRAFT_3` in which every revision still fails, and confirm the report is rendered with the "Blocked by the provenance audit" banner instead of being published.
> 3. Add a sixth source that re-reports S1's 3.2 billion figure with `derived_from="S1"`; the 2024 analyst figure should still count as one origin.

### Run it for real (live)

The doc-analysis subagent (Citations API + structured enrichment) and the synthesis run on the real model; the web-research and archive subagents keep their recorded responses, so the report stays comparable and no web searches are billed. Claim IDs and wording will differ from the offline run; the guarantees (cited evidence, audit, coverage) do not.

In [ ]:
# (live) doc-analysis + synthesis on the real model
live_reports = run_subagents(client, ScriptedClaude(WEB_REPLY, WEB_FIELDS).client)
live_state = coordinate(live_reports, store)
print("claims:", len(store.get(live_state["ref"], ReportClaim)), "| conflicts:", [(k["conflict_type"], k["status"]) for k in live_state["conflicts"]])
print("unsupported numeric text from the model:", live_state["unsupported"])
live_narrative, live_problems = write_with_audit(client, live_state)
display(Markdown(render_report(live_narrative, live_state, live_problems)))

## Recap

- Attribution is lost at **every summarizing hop**; the fix is a claim record (statement + verbatim excerpt + source + location + dates + method) that every agent passes on as data, with required-but-nullable metadata.
- Conflicts: normalize, classify (definition, temporal, undetermined, methodology, genuine), keep **both values with attribution**, and let the **coordinator** reconcile with a recorded reason.
- Dates: data period ≠ publication date ≠ retrieval date. Resolve relative dates against the publication date; sort by period; freshness is a policy in code.
- Rendering: tables for numbers, prose for news, lists for technical findings, chronology for time series, a contested section for conflicts. Code renders numbers and references; the model writes by claim ID.
- Coverage per sub-question with `covered` / `empty` / `failed` / `not_searched`; support levels from independent origins, not self-rated confidence.
- Citations API: valid pointers by construction, `cited_text` free of output tokens, all-or-none, **no structured outputs in the same request** (two steps). `search_result` blocks bring the same to RAG (block = citable unit). Web search citations are always on; web fetch citations are opt-in; send encrypted fields back unchanged.
- Verify before publishing: code audit first, fresh-instance review next, humans for high-stakes or contested decisions.

Next: [homework](02_homework.ipynb)